# RSNA Knee — KneeMind v12 (`PRESET="evidence"`, v11 recipe + 2 more members, longer training)

**v11 result (real run):** hidden score **0.908** (v7 0.897). Two members, 18 epochs each, gold-58 AUC 0.862 / 0.876 (still rising at the last epoch).
An audit of the v11 code found **no bug that corrupts predictions** (train/cache/inference preprocessing match, ids and column order are validated,
decode failures are masked, not fed as black frames). What it did find, and what v12 changes:

| Finding in v11 | v12 |
|---|---|
| Only 2 members; their gold predictions correlate 0.85-0.95 per finding, so averaging more members still helps | `ENSEMBLE_WITH` defaults to `knee_v11`: the 2 new v12 members are averaged with the 2 attached v11 members (same preprocessing, checked at load time) |
| Gold AUC still rising at epoch 18 | `EPOCHS` 18 -> 24 for the new members (~4.5 h per round on T4 x2) |
| Both v11 members would be re-trained identically (same seeds 42, 43) | `SEED0=100`: new members use seeds 142, 143 (env `KNEE_SEED0`) |
| Saved weights were cast to fp16, but the OOF/gold numbers were computed with fp32 | weights saved in **fp32** |
| Two members train in two threads and each called `torch.manual_seed`, which reseeds *every* GPU: the other member's generator could be reset mid-setup | each member seeds its own CPU and GPU generator under a lock |
| `N_MEMBERS` could not be changed without editing the notebook | env `KNEE_MEMBERS` |
| Ensemble gold AUC and the blend gate only looked at this version's `oof_*.csv` | they now also read the attached v11 `oof_v11_*.csv` |

**Not changed on purpose:** label fusion, loss, augmentation, resolution. Their effect cannot be measured with the 58 gold studies (SE about 0.02).
**Cost:** inference runs 4 models instead of 2, so the hidden run takes roughly twice as long on the GPU part. If the efficiency track matters more
than the last 0.005 AUC, set `KNEE_ENSEMBLE_WITH=""` (own members only) or `KNEE_MEMBERS=1`.

**Run:** attach the v11 cache output (same cache tag, no re-caching) and the v11 pack, `MODE="full"`, GPU T4 x2, Internet ON. Then `MODE="infer"`
with the v12 output **and** the v11 pack attached, Internet OFF.

---

# RSNA Knee — KneeMind v11 (`PRESET="evidence"`)

v7 (0.897) and v8 (0.860) were tuned by guesswork. v11 changes only what **someone else's pre-registered A/B tests in this same competition**
(public repo TranBaDat2607/RSNA-Knee-Abnormality-Detection, docs/phase2_notes.md) found to matter for a small ConvNeXt trained on report labels, which
took their own models from the 0.89-0.90 class to **0.92 LB** (R5t 0.922, plus two full-data models 0.924):

| Lever (their evidence) | v7 | v11 `evidence` |
|---|---|---|
| Training length: 20 vs 12 epochs +0.0045 val; every run still improving at its last epoch | 10 (9 done) | **18** |
| Resolution: 336 px vs 256 px +0.0025 val (MCL, lateral meniscus) | 256 px | **336 px** at the same 153.6 mm field of view |
| Central slice span: 4-96 % lost 0.0055 against 15-85 % | 0-100 % | **15-85 %** |
| Windows per step: 16 random windows beat 40 (2.4x cheaper) | all 60 every step | **20 random windows per step** (4 per view), 30 per view-set at test time |
| Backbone: ConvNeXt-tiny, ResNet-50 and CoAtNet-1 no better than nano | tiny | **ConvNeXt-nano** (in12k) |
| Positive weighting `clip((1-p)/p, 1, 5)` in the BCE | none | **on** |
| Mirroring left knees to one orientation: the side tag exists on only ~50 % of series; their A/B of canonical windows was clearly negative (-0.10 macro after 3 epochs, mechanism unclear) | on | **off** (`KNEE_CANON=1` restores it; cache tag gets `_raw`) |
| Report labels only: mixing out-of-fold predictions into targets *hurt* (LB 0.922 -> 0.920 -> 0.917; gold falls as more is mixed) | v10 had it on | **off** (`REFINE`, `USE_DISTILL` default 0) |

The same evidence says the remaining image-side levers are small (every backbone/layout variant lands within about 0.005, models agree at rank
correlation ~0.97): the models are label-bound. Do not expect more than the lift above from this recipe.

**Status (v11): run; hidden score 0.908.** Their pipeline differs from this one (public 44-slice corpus,
MIL attention head, no study transformer), so the lift may not transfer in full.

Files are `knee_v11_*`. A v7 cache cannot be reused (different pixels), so build the cache first (`MODE="cache"` in CPU sessions).

---

## Inherited from v7
# RSNA Knee — KneeMind v7: stronger reader, cleaner labels, all-data training, gated stack blend

**What changed from v6 (LB 0.881)**
1. **`PRESET="strong"` (new default).** ConvNeXt-Tiny (ImageNet-22k) at 256 px / 0.6 mm, 24 cached / 12 used slices per view,
   10 epochs. v6 `"fast"` used ConvNeXt-Nano at 224 px; public write-ups gained most from resolution and training length.
2. **`TRAIN_SCHEME="all_weak"`.** The 58 gold studies are the only honest validation set and are never trained on, so a
   held-out weak fold only costs data. Each model now trains on **all 4,349 weak studies** (+33% per model vs 4 folds),
   two seeds at once (one per T4), keeping the final EMA weights (no selection on the noisy 58-study gold AUC).
3. **Leak-guarded label fusion.** Attach more public label tables (e.g. `stevenleehans/rsna-knee-llm-report-labels`).
   Several public tables copy the 58 gold labels verbatim, which would hand them all the weight in v6's fusion; v7 detects
   that and gives such a table an average weight instead.
4. **Large caches go to `/tmp`.** The `"strong"` cache (~17 GB) is too close to Kaggle's 20 GB output limit; build it in
   CPU-only `MODE="cache"` sessions (no GPU quota) and attach those outputs, or let the GPU run cache into `/tmp`.
5. **Gated stack blend.** With a public stack in the same notebook, the blend weight is chosen on the 58 gold studies
   (single global weight, grid 0–0.3) and stays **0** unless it improves gold AUC by ≥ `GATE_MIN_GAIN`. A weak member can
   no longer silently pull the stack down.
6. Weights/config are named `knee_v7_*`, so v6 weights (different preprocessing) are never mixed in by accident.

**How to run**

| Run | `MODE` | Settings | Attach (Add Input) | Output |
|---|---|---|---|---|
| **0** (optional, CPU, free) | `"cache"`, `SHARD = 0..3` | CPU, Internet ON | competition data | `knee_cache_s{SHARD}_*` (one quarter of the cache each) |
| **1** | `"full"` | GPU **T4 ×2**, Internet **ON**, *Save & Run All* | competition data + label datasets (+ the 4 run-0 outputs) | `knee_v7_s0.pt`, `knee_v7_s1.pt`, config, `wheels/`, gold OOF, `kneemind_pack.zip` |
| **2** | `"infer"` | GPU, Internet **OFF** | competition data + run 1's output | `submission.csv` → **Submit** |

Blending with a public stack: see the section before the last cell.

In [ ]:
import os, sys, re, json, math, time, glob, random, warnings, subprocess, unicodedata, gc
# Kaggle T4 x2: GPU peer-to-peer transfers can deadlock multi-GPU code silently. Route through host memory instead.
os.environ.setdefault("NCCL_P2P_DISABLE", "1"); os.environ.setdefault("NCCL_IB_DISABLE", "1")
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ============================== WHAT TO RUN ==============================
MODE = "full"             # "profile": ~8 min on GPU, times GPU step / data loader / DICOM decode, no training
                          # "full"  : ONE GPU session, Internet ON -> cache + train folds + test check (run 1)
                          # "infer" : Internet OFF, attach run 1's output -> Submit (run 2)
                          # advanced, split across sessions: "cache" (per SHARD) and "train" (per FOLD)
PRESET = "evidence"       # "evidence": v11 default, see README_v11.md
                          # "lean"  : v8 EfficientNet-B0 320 px (scored 0.860: worse than v7, do not use)
                          # "strong": v7 / v9 default: ConvNeXt-Tiny 256 px, all weak data, 2 members trained side by side (~5-7 h)
                          # "fast"  : v6 settings (ConvNeXt-Nano 224 px), ~2.5-3 h
                          # "best"  : v6 4-fold ConvNeXt-Tiny 256 px, ~6-8 h+
TRAIN_SCHEME = "all_weak" # "all_weak": every model trains on ALL weak studies, validated on the 58 gold only
                          # "folds"   : v6 behaviour (held-out weak fold + gold)
N_MEMBERS = int(os.environ.get("KNEE_MEMBERS", 2))   # all_weak: models (seeds) to train; 2 = one per T4, trained at the same time
SELECT_LAST = True        # all_weak: keep the final EMA weights instead of the best-gold epoch (58 studies are too noisy to select on)
FULL_FOLDS = [0, 1, 2, 3] # full: folds trained back to back; a fold only starts if it can finish in time
SHARD, N_SHARDS = 0, 4    # cache mode only
FOLD, N_FOLDS = 0, 4      # train mode only (N_FOLDS also defines the split in full mode)
FINAL_ALL_DATA = False    # train: after folds look good, train on ALL weak studies + gold (fixed EPOCHS, no validation)
USE_DISTILL = os.environ.get("KNEE_DISTILL", "0") == "1"   # v11: OFF by default (heavier OOF mixing lowered the LB in public A/Bs); only acts when oof_v*_f*.csv (e.g. the v6 pack) is attached;
#                           KNEE_DISTILL=0 disables.   # train: mix out-of-fold predictions (oof_v6_f*.csv attached) into the weak targets
DISTILL_W = 0.3

# ---- v9: denoise the weak labels while training (temporal ensembling of the model's own predictions) ----
REFINE = os.environ.get("KNEE_REFINE", "0") == "1"   # v11: OFF (pseudo-label mixing did not help in public A/Bs); weak studies only; the 58 gold studies are never touched
REFINE_ALPHA = 0.35      # largest share of the target taken from the memory (scaled per finding by label noise, see below)
REFINE_BETA = 0.7        # memory update  m <- BETA*m + (1-BETA)*p : averages roughly the last 3 epochs of predictions
REFINE_FROM = 1          # epoch index when the memory starts absorbing predictions (epoch 0 is warm-up: predictions are junk)
REFINE_START, REFINE_RAMP = 2, 3   # target mix is 0 before this epoch, then ramps linearly to full strength over REFINE_RAMP epochs
T_TRAIN = int(os.environ.get("KNEE_TTRAIN", 0))   # v11: slices per view per training step (0 = use T_SLICES, as v7); eval always uses T_SLICES
POS_W_MAX = float(os.environ.get("KNEE_POSW", 1.0))   # v11: BCE pos_weight = clip((1-prevalence)/prevalence, 1, POS_W_MAX); 1.0 = off (v7)
DROP_PATH = 0.2          # v11: stochastic depth of the encoder (the evidence preset lowers it)
DECODE_MAX_BAD = 0.4     # v10: a series with more than this fraction of undecodable sampled slices is treated as missing (v7 fed black frames)
REFINE_FLOOR = 0.25      # even a finding with clean-looking labels keeps this fraction of REFINE_ALPHA (58-study AUCs are noisy)
# infer: also load these earlier weight sets (same preprocessing required, checked at load time): free ensemble members
ENSEMBLE_WITH = [s.strip() for s in os.environ.get("KNEE_ENSEMBLE_WITH", "knee_v11").split(",") if s.strip()]   # v12: the v11 members join the ensemble (KNEE_ENSEMBLE_WITH="" = own members only)
SEED0 = int(os.environ.get("KNEE_SEED0", 100))   # v12: member seeds are 42 + SEED0 + index, so they differ from v11 (42, 43)

# When you press "Submit", Kaggle re-runs this notebook on the hidden test set with this variable set.
# Whatever MODE the notebook was saved with, the re-run must only do inference.
MODE = os.environ.get("KNEE_MODE", MODE); PRESET = os.environ.get("KNEE_PRESET", PRESET)   # env overrides (testing)
TRAIN_SCHEME = os.environ.get("KNEE_SCHEME", TRAIN_SCHEME); SHARD = int(os.environ.get("KNEE_SHARD", SHARD))
VER = "v12"; WPFX = f"knee_{VER}"   # weight / config / OOF file prefix (v6 files are ignored)
IS_RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
if IS_RERUN: MODE = "infer"

# ============================== PATHS ==============================
INPUT_ROOT = Path(os.environ.get("KNEE_INPUT_ROOT", "/kaggle/input"))
COMP_DIR = Path(os.environ.get("KNEE_COMP_DIR", str(INPUT_ROOT / "competitions/rsna-knee-abnormality-detection")))
if not COMP_DIR.exists():
    COMP_DIR = INPUT_ROOT / "rsna-knee-abnormality-detection"
WORK = Path(os.environ.get("KNEE_WORK", "/kaggle/working")); WORK.mkdir(parents=True, exist_ok=True)
# full mode caches into /kaggle/working, so it is SAVED with the notebook output: attach that output next time and
# caching is skipped. The folder name records the preprocessing, so a "fast" cache is never reused for "best".
SKIP_DIRS = {"competitions", "rsna-knee-abnormality-detection", "train_series", "test_series"}

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

# ============================== PREPROCESSING (cache == inference, must match) ==============================
# (plane, preferred Fluid_Sensitive). Sag-FS / Cor-FS / Ax-FS carry fluid, marrow, ligaments, menisci;
# Sag-T1/PD and Cor-T1/PD add fracture lines, OA osteophytes and meniscal morphology.
VIEWS = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1), ("Sagittal", 0), ("Coronal", 0)]
NV = len(VIEWS)
SPACING_MM = 0.6          # in-plane resampling target
IMG = int(os.environ.get("KNEE_IMG", 256))   # center crop -> 153.6 mm field of view
DEPTH = int(os.environ.get("KNEE_DEPTH", 24))  # slices stored per series (evenly spaced along the slice normal)
SPAN = (0.0, 1.0)         # fraction of the position-sorted slice stack that is sampled (v8 "lean": 0.06-0.94)
CANONICALIZE_LATERALITY = True   # mirror left knees -> medial/lateral always on the same image side

# ============================== MODEL / TRAINING ==============================
BACKBONE = os.environ.get("KNEE_BACKBONE", "convnext_tiny.fb_in22k_ft_in1k")   # alt: "tf_efficientnetv2_s.in21k_ft_in1k"
PRETRAINED = os.environ.get("KNEE_PRETRAINED", "1") == "1"
T_SLICES = int(os.environ.get("KNEE_T", 12))   # slices per view fed to the model (sampled from DEPTH)
DIM, MIX_LAYERS, HEADS = 384, 2, 6
EPOCHS = int(os.environ.get("KNEE_EPOCHS", 10))
BATCH_SIZE, ACCUM = 2, 8                         # effective batch 16
LR_ENC, LR_HEAD, WD = 1e-4, 5e-4, 0.05
WARMUP_EPOCHS = 1.0
EMA_DECAY = 0.997
GRAD_CKPT = True
VIEW_DROP = 0.15          # randomly hide a view during training (robust to missing series at test time)
GOLD_WEIGHT = 3.0         # only used when FINAL_ALL_DATA
TIME_BUDGET_H = float(os.environ.get("KNEE_BUDGET_H", 11.2))   # whole session incl. caching (Kaggle kills at 12 h)
FINAL_RESERVE_H = 0.25    # kept free at the end of a full run for the test check + saving outputs
INFER_BUDGET_H = 8.0      # Kaggle allows 9 h for the hidden run; TTA switches off if the projection exceeds 85% of this
N_TTA = 2                 # inference: two interleaved slice sets
W_MINE = None             # rank-blend weight when stack_submission.csv exists: None = choose on the 58 gold studies
                          # (needs stack_gold_preds.csv attached, else 0); a number forces that weight
W_GRID = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
GATE_MIN_GAIN = 0.003     # blend only if it beats the stack alone on gold by at least this much

# ============================== PRESET ==============================
# "fast" cuts compute ~3x: 224 px (0.7 mm, same 156 mm field of view), 16 cached / 10 used slices per view,
# ConvNeXt-Nano (ImageNet-12k), 8 epochs, 2 folds. Environment variables still override (used for testing).
if PRESET == "strong":
    SPACING_MM = 0.6
    IMG = int(os.environ.get("KNEE_IMG", 256))
    DEPTH = int(os.environ.get("KNEE_DEPTH", 24))
    T_SLICES = int(os.environ.get("KNEE_T", 12))
    BACKBONE = os.environ.get("KNEE_BACKBONE", "convnext_tiny.fb_in22k_ft_in1k")
    EPOCHS = int(os.environ.get("KNEE_EPOCHS", 10))
    TIME_BUDGET_H = float(os.environ.get("KNEE_BUDGET_H", 9.0))   # whole session (caching + training + saving). v7 used 5.8, which can cut epochs: watch for "!!" lines
    GRAD_CKPT = False     # v10: faster steps; switches itself ON if the GPU runs out of memory (see train_fold)
elif PRESET == "evidence":
    # v11: only changes that had pre-registered evidence in a public A/B study (see README_v11.md)
    SPACING_MM = float(os.environ.get("KNEE_SPACING", 0.457))     # 336 px x 0.457 mm = 153.6 mm field of view, same as v7
    IMG = int(os.environ.get("KNEE_IMG", 336))
    DEPTH = int(os.environ.get("KNEE_DEPTH", 12))                 # slices cached per view over the central span
    SPAN = (0.15, 0.85)
    CANONICALIZE_LATERALITY = os.environ.get("KNEE_CANON", "0") == "1"   # E5b: mirroring on a ~50%-reliable side tag hurt (-0.10 macro); raw windows + the side tag as an input instead
    T_SLICES = int(os.environ.get("KNEE_T", 6))                   # eval: 5 views x 6 = 30 windows per pass, 2 interleaved passes
    T_TRAIN = int(os.environ.get("KNEE_TTRAIN", 4))               # train: 5 views x 4 = 20 random windows per study per step
    BACKBONE = os.environ.get("KNEE_BACKBONE", "convnext_nano.in12k_ft_in1k")
    EPOCHS = int(os.environ.get("KNEE_EPOCHS", 24))   # v12: v11 gold AUC was still rising at epoch 18 (cosine schedule is stretched to match)
    BATCH_SIZE, ACCUM = 4, 4                                      # effective batch 16, as v7
    LR_ENC = 1.5e-4
    DROP_PATH = 0.05
    POS_W_MAX = float(os.environ.get("KNEE_POSW", 5.0))
    TIME_BUDGET_H = float(os.environ.get("KNEE_BUDGET_H", 9.0))
    GRAD_CKPT = False     # nano at 336 px, 20 windows x 4 studies: fits a T4 in fp16; switches itself on after an out-of-memory error
elif PRESET == "lean":
    # v8: finer pixels, a cheaper backbone, and slices concentrated on the joint.
    # * 0.48 mm x 320 px = 153.6 mm field of view (same as v7), 20% finer than v7 (meniscus/MCL are small structures)
    # * SPAN 6-94 %: the Raptor study found 6-94 % beat 2-98 % in 95 % of paired resamples (+0.005, suggestive)
    # * 4 views: the second (non-fluid) coronal series got 1.5-6 % of attention in the public CoAtNet and removing the
    #   non-fluid series together cost 0.000 there; 4 views x 10 slices = 40 windows (>= 24 needed for full accuracy)
    SPACING_MM = 0.48
    IMG = int(os.environ.get("KNEE_IMG", 320))
    DEPTH = int(os.environ.get("KNEE_DEPTH", 20))
    T_SLICES = int(os.environ.get("KNEE_T", 10))
    VIEWS = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1), ("Sagittal", 0)]; NV = len(VIEWS)
    SPAN = (0.06, 0.94)
    BACKBONE = os.environ.get("KNEE_BACKBONE", "tf_efficientnet_b0.ns_jft_in1k")
    EPOCHS = int(os.environ.get("KNEE_EPOCHS", 12))
    LR_ENC = 1.5e-4       # an EfficientNetV2-S reader trained stably at 1.5e-4 with 1 warm-up epoch; 3 of 4 folds diverged higher
    TIME_BUDGET_H = float(os.environ.get("KNEE_BUDGET_H", 5.8))
    GRAD_CKPT = False     # small model; auto re-enabled on out-of-memory
elif PRESET == "fast":
    SPACING_MM = 0.7
    IMG = int(os.environ.get("KNEE_IMG", 224))
    DEPTH = int(os.environ.get("KNEE_DEPTH", 16))
    T_SLICES = int(os.environ.get("KNEE_T", 10))
    BACKBONE = os.environ.get("KNEE_BACKBONE", "convnext_nano.in12k_ft_in1k")
    EPOCHS = int(os.environ.get("KNEE_EPOCHS", 8))
    FULL_FOLDS = FULL_FOLDS[:2]
    TIME_BUDGET_H = float(os.environ.get("KNEE_BUDGET_H", 6.0))   # safety cap; normally finishes well before
    GRAD_CKPT = False     # small model fits in memory without it: ~25% faster steps (auto re-enabled on out-of-memory)
NUM_WORKERS = min(os.cpu_count() or 4, 8)
CACHE_PROCS = min(2 * (os.cpu_count() or 4), 12)   # caching waits on disk reads too, so use more processes than CPUs
PARALLEL_FOLDS = True     # with 2 GPUs: train two folds at the same time, one per GPU (no GPU-to-GPU traffic)
VAL_WEAK_MAX = 400        # per-epoch validation: 58 gold + this many weak studies; full OOF computed once at the end
CACHE_TAG = f"{IMG}px_{DEPTH}d_{SPACING_MM}mm_{len(VIEWS)}v"
# A different slice span means different pixels: it must never share a cache with another span. (v7 tags are unchanged.)
if tuple(SPAN) != (0.0, 1.0): CACHE_TAG += f"_sp{int(round(SPAN[0] * 100))}-{int(round(SPAN[1] * 100))}"
# A cache bigger than ~14 GB cannot be saved with the notebook output (20 GB limit, weights must fit too):
# build it in /tmp instead (lost after the session), or better, in CPU-only MODE="cache" sessions.
if not CANONICALIZE_LATERALITY: CACHE_TAG += "_raw"   # un-mirrored pixels never share a cache with mirrored ones (v7 tags unchanged)
CACHE_EST_GB = 4407 * NV * DEPTH * IMG * IMG * 0.5 / 1e9   # ~50% npz compression
_cache_root = WORK if CACHE_EST_GB < 14 else Path("/tmp")
LOCAL_CACHE = Path(os.environ.get("KNEE_LOCAL_CACHE", str(_cache_root / f"knee_cache_sall_{CACHE_TAG}")))
SEED = 42 + FOLD
random.seed(SEED); np.random.seed(SEED)
T0 = time.time()
def log(*a):
    """Timestamped progress line, flushed immediately so you always see the last stage reached."""
    print(f"[{(time.time() - T0) / 60:6.1f} min]", *a, flush=True)
print(f"PRESET={PRESET}: {IMG}px @ {SPACING_MM} mm, {DEPTH} cached / {T_SLICES} used slices, {BACKBONE}, {EPOCHS} epochs, "
      + (f"all weak data, {N_MEMBERS} members" if TRAIN_SCHEME == "all_weak" else f"folds {FULL_FOLDS}"))
print(f"cache estimate ~{CACHE_EST_GB:.0f} GB -> {LOCAL_CACHE}")
print(f"MODE={MODE} (hidden re-run: {IS_RERUN}) SHARD={SHARD} FOLD={FOLD} | COMP_DIR exists: {COMP_DIR.exists()} | CPUs: {os.cpu_count()}")

## 1. DICOM decoders
Online sessions install `pylibjpeg` / `gdcm` and save the wheels to `wheels/`; the offline inference session installs them from the attached training output.

In [ ]:
def have_internet():
    import socket
    try:
        socket.create_connection(("pypi.org", 443), timeout=3).close(); return True
    except OSError:
        return False

PACK_NAME = "kneemind_pack.zip"
PACK_DIR = Path(os.environ.get("KNEE_PACK_DIR", "/tmp/kneemind_pack"))   # where attached packs get unzipped

def walk_inputs(max_depth=6):
    """Yield (dirpath, dirnames, filenames) under INPUT_ROOT (skipping the huge competition data) and PACK_DIR."""
    for root in (INPUT_ROOT, PACK_DIR):
        if not root.exists(): continue
        for dirpath, dirnames, filenames in os.walk(root):
            depth = len(Path(dirpath).relative_to(root).parts)
            dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
            if depth >= max_depth: dirnames[:] = []
            yield dirpath, dirnames, filenames

def unpack_packs():
    """Unzip every attached kneemind_pack.zip (weights + config + wheels) so inference can use it offline.
    Kaggle keeps zips zipped in notebook outputs, but unzips them when you upload a dataset; both cases work."""
    import zipfile
    if not INPUT_ROOT.exists(): return []
    found = []
    for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith("knee_cache_s")]
        for fn in sorted(f for f in filenames if re.match(r"kneemind_pack.*\.zip$", f)):   # v10: renamed packs (kneemind_pack_v7_2.zip) too
            src = Path(dirpath) / fn
            dst = PACK_DIR / f"pack{len(found)}"
            if not dst.exists():
                with zipfile.ZipFile(src) as z: z.extractall(dst)
            found.append(src)
    for p in found: print("unpacked", p, "->", PACK_DIR)
    return found

if MODE == "infer": unpack_packs()

def find_in_inputs(name):
    for dirpath, dirnames, filenames in walk_inputs():
        if name in filenames or name in dirnames: return Path(dirpath) / name
    return None

PKGS = ["pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "python-gdcm"]
FORCE_OFFLINE = os.environ.get("KNEE_FORCE_OFFLINE") == "1"           # for testing the offline path
ONLINE = (not FORCE_OFFLINE) and have_internet()
WHEEL_DIR = WORK / "wheels"
if os.environ.get("KNEE_SKIP_PIP") != "1":
    pip = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    if ONLINE:
        # install now, and save wheels so the offline submission run can install the exact same packages
        subprocess.run(pip + PKGS + ["timm"], check=False)
        WHEEL_DIR.mkdir(exist_ok=True)
        # --no-deps: numpy / torch / huggingface-hub are already on the Kaggle image; only the packages themselves (~30 MB)
        subprocess.run([sys.executable, "-m", "pip", "download", "-q", "--disable-pip-version-check", "--no-deps",
                        "-d", str(WHEEL_DIR)] + PKGS + ["timm"], check=False)
        print("online: installed decoders + timm; wheels saved to", WHEEL_DIR, f"({len(list(WHEEL_DIR.glob('*.whl')))} files)")
    else:
        wdir = find_in_inputs("wheels")
        if wdir is not None:
            need = PKGS + ([] if __import__("importlib").util.find_spec("timm") else ["timm"])
            r = subprocess.run(pip + ["--no-index", "--no-deps", "--find-links", str(wdir)] + need, capture_output=True, text=True)
            print("offline: installed", need, "from", wdir, "| ok" if r.returncode == 0 else "| pip said:\n" + r.stderr[-1500:])
        else:
            print("WARNING offline and no 'wheels' folder attached: JPEG-compressed DICOMs may not decode")

import pydicom, cv2
cv2.setNumThreads(0)      # parallelism comes from worker processes; OpenCV threads inside them only oversubscribe the CPUs

def decoder_status():
    st = {}
    for mod in ("pylibjpeg", "libjpeg", "openjpeg", "gdcm"):
        try: __import__(mod); st[mod] = True
        except Exception: st[mod] = False
    return st
DECODERS = decoder_status()
print("internet:", ONLINE, "| DICOM decoders:", DECODERS)
if MODE in ("infer", "full") and not (DECODERS["gdcm"] or (DECODERS["libjpeg"] and DECODERS["openjpeg"])):
    print("WARNING: no JPEG Lossless / JPEG 2000 decoder available. Attach the training output that contains 'wheels/'.")
print("pydicom", pydicom.__version__, "| cv2", cv2.__version__)

## 2. Series selection and DICOM → canonical, fixed-spacing volume
Every series becomes `[DEPTH, IMG, IMG]` uint8: slices sorted along the true slice normal, resampled to 0.6 mm pixels,
center-cropped to 153.6 mm, windowed per series (1st–99.5th percentile of non-zero voxels), oriented so the in-plane
axes point toward the patient's +axes, and left knees mirrored onto right-knee layout.
**The same function runs at cache time and at test time**, so train and test preprocessing cannot drift apart.

In [ ]:
def pick_series(sdf):
    """One series per VIEW: exact (plane, FS) match first, then the same plane with the other FS flag; never reuse a series."""
    out, used = {}, set()
    sdf = sdf.assign(_plane=sdf["Anatomical_Plane"].astype(str).str.lower())
    fs_col = "Fluid_Sensitive" if "Fluid_Sensitive" in sdf.columns else "Fat_Suppression"
    for v_i, (plane, fs) in enumerate(VIEWS):
        p = sdf[(sdf._plane == plane.lower()) & (~sdf.SeriesInstanceUID.astype(str).isin(used))]
        if len(p) == 0: continue
        p = p.assign(_score=(pd.to_numeric(p[fs_col], errors="coerce").fillna(0) == fs).astype(int))
        uid = str(p.sort_values("_score", ascending=False, kind="stable").iloc[0].SeriesInstanceUID)
        out[v_i] = uid; used.add(uid)
    return out

_LAT_L = re.compile(r"\b(LEFT|LT|LINKS|IZQ\w*|GAUCHE|SINIST\w*|ESQ\w*)\b|(^|[\s_\-])L($|[\s_\-])")
_LAT_R = re.compile(r"\b(RIGHT|RT|RECHTS|DERECH\w*|DROIT\w*|DESTR\w*|DIREIT\w*)\b|(^|[\s_\-])R($|[\s_\-])")
def get_laterality(ds):
    for tag in ("Laterality", "ImageLaterality"):
        v = str(getattr(ds, tag, "") or "").upper().strip()
        if v in ("L", "R"): return v
    txt = " ".join(str(getattr(ds, t, "") or "") for t in
                   ("SeriesDescription", "StudyDescription", "BodyPartExamined", "ProtocolName")).upper()
    l, r = bool(_LAT_L.search(txt)), bool(_LAT_R.search(txt))
    return "L" if (l and not r) else "R" if (r and not l) else "U"

def _pixel_spacing(ds):
    for tag in ("PixelSpacing", "ImagerPixelSpacing"):
        try:
            v = [float(x) for x in getattr(ds, tag)]
            if len(v) == 2 and v[0] > 0 and v[1] > 0: return v
        except Exception:
            pass
    return [SPACING_MM, SPACING_MM]

def _to_frame(a, ds, sp):
    """float32 2D frame -> resampled to SPACING_MM, center crop/pad to IMG."""
    a = a.astype(np.float32)
    try: a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    except Exception: pass
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a
    h, w = a.shape
    nh, nw = max(8, int(round(h * sp[0] / SPACING_MM))), max(8, int(round(w * sp[1] / SPACING_MM)))
    a = cv2.resize(a, (nw, nh), interpolation=cv2.INTER_AREA if nh < h else cv2.INTER_LINEAR)
    out = np.zeros((IMG, IMG), np.float32)
    y0, x0 = (nh - IMG) // 2, (nw - IMG) // 2
    sy, sx = max(y0, 0), max(x0, 0)
    dy, dx = max(-y0, 0), max(-x0, 0)
    hh, ww = min(IMG - dy, nh - sy), min(IMG - dx, nw - sx)
    out[dy:dy + hh, dx:dx + ww] = a[sy:sy + hh, sx:sx + ww]
    return out

def _span_idx(n, depth):
    """`depth` evenly spaced slice indices over the central SPAN of an n-slice stack (SPAN read at call time)."""
    return np.linspace(SPAN[0] * (n - 1), SPAN[1] * (n - 1), depth).round().astype(int)

def load_volume(series_dir, depth=None):
    """-> (uint8 [depth, IMG, IMG], laterality) or (None, 'U')."""
    depth = depth or DEPTH   # read at call time: inference loads DEPTH from the model config, which may differ from the preset
    files = sorted(Path(series_dir).glob("*.dcm"))
    metas = []
    for f in files:
        try: metas.append((f, pydicom.dcmread(str(f), stop_before_pixels=True)))
        except Exception: pass
    if not metas: return None, "U"
    shp = lambda ds: (int(getattr(ds, "Rows", 0) or 0), int(getattr(ds, "Columns", 0) or 0))
    main = Counter(shp(ds) for _, ds in metas).most_common(1)[0][0]
    metas = [m for m in metas if shp(m[1]) == main]
    ds0 = metas[0][1]
    lat, sp = get_laterality(ds0), _pixel_spacing(ds0)

    row = col = normal = None
    try:
        iop = np.array(ds0.ImageOrientationPatient, dtype=float)
        row, col = iop[:3], iop[3:]
        normal = np.cross(row, col)
        if normal[np.argmax(np.abs(normal))] < 0: normal = -normal
    except Exception:
        pass

    frames = []
    n_frames = int(getattr(ds0, "NumberOfFrames", 1) or 1)
    if len(metas) == 1 and n_frames > 1:                       # enhanced multi-frame DICOM
        ds = pydicom.dcmread(str(metas[0][0])); arr = ds.pixel_array
        if arr.ndim == 4: arr = arr[..., 0]
        for i in _span_idx(arr.shape[0], depth):
            frames.append(_to_frame(arr[i], ds, sp))
    else:
        def key(ds):
            if normal is not None:
                try: return float(np.dot(normal, np.array(ds.ImagePositionPatient, dtype=float)))
                except Exception: pass
            try: return float(ds.InstanceNumber)
            except Exception: return 0.0
        metas.sort(key=lambda m: key(m[1]))
        cache, badset = {}, set()
        for i in _span_idx(len(metas), depth):
            if i not in cache:
                try:
                    ds = pydicom.dcmread(str(metas[i][0])); a = ds.pixel_array
                    if a.ndim == 3:
                        a = a[..., 0] if a.shape[-1] in (3, 4) else a[a.shape[0] // 2]
                    cache[i] = _to_frame(a, ds, sp)
                except Exception:
                    cache[i] = np.zeros((IMG, IMG), np.float32); badset.add(i)
            frames.append(cache[i])
        if len(badset) > DECODE_MAX_BAD * max(len(cache), 1):      # v10: never feed a mostly-black volume as a real view
            raise RuntimeError(f"{len(badset)}/{len(cache)} sampled slices could not be decoded in {series_dir}")
    vol = np.stack(frames)
    nz = vol[vol > 0]
    lo, hi = np.percentile(nz if nz.size > 100 else vol, (1, 99.5))
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1) * 255

    if row is not None:
        if row[np.argmax(np.abs(row))] < 0: vol = vol[:, :, ::-1]
        if col[np.argmax(np.abs(col))] < 0: vol = vol[:, ::-1, :]
        if CANONICALIZE_LATERALITY and lat == "L":
            axes = {0: normal, 1: col, 2: row}
            ax = max(axes, key=lambda k: abs(axes[k][0]))      # the axis that runs along patient left-right
            vol = np.flip(vol, axis=ax)
    return np.ascontiguousarray(vol).astype(np.uint8), lat

_DEC_WARN = [0]
def _decode_warn(study_uid, series_uid, err):
    _DEC_WARN[0] += 1                                  # per worker process; only the first few are printed
    if _DEC_WARN[0] <= 5: print(f"DECODE FAILURE study {str(study_uid)[-12:]} series {str(series_uid)[-12:]}: {repr(err)[:160]} -> view masked out", flush=True)

def process_study(study_uid, sdf, root):
    """-> vol [NV, DEPTH, IMG, IMG] uint8, view mask [NV], laterality one-hot [3] (L, R, unknown)."""
    chosen = pick_series(sdf)
    vol = np.zeros((NV, DEPTH, IMG, IMG), np.uint8)
    vmask = np.zeros(NV, np.float32); lats = []
    for v in range(NV):
        if v not in chosen: continue
        try: arr, lat = load_volume(Path(root) / str(study_uid) / chosen[v])
        except Exception as e: arr, lat = None, "U"; _decode_warn(study_uid, chosen[v], e)
        if arr is not None:
            vol[v] = arr; vmask[v] = 1; lats.append(lat)
    known = [l for l in lats if l != "U"]
    lat = Counter(known).most_common(1)[0][0] if known else "U"
    return vol, vmask, np.array([lat == "L", lat == "R", lat == "U"], np.float32)

## 3. Cache: every training study → one `.npz` (5 views × 24 slices × 256²), built once with all CPU cores
* `full` mode: all 4,407 studies go to `/kaggle/working/knee_cache_sall_<settings>/` (saved with the output),
  roughly 45-60 min. Studies already cached (this session, or an attached earlier output) are skipped.
* `cache` mode (optional, for splitting work across sessions): one shard goes to `/kaggle/working`, to be saved as a Dataset.

In [ ]:
from multiprocessing import Pool
from tqdm.auto import tqdm
_SERIES, _OUT_DIR = {}, None

def _cache_one(uid):                       # top-level so the worker pool can pickle it
    out = _OUT_DIR / f"{uid}.npz"
    if out.exists(): return uid, -1
    sdf = _SERIES.get(uid)
    if sdf is None: return uid, 0
    try:
        vol, vmask, lat = process_study(uid, sdf, COMP_DIR / "train_series")
        tmp = _OUT_DIR / f"{uid}.tmp.npz"
        np.savez_compressed(tmp, vol=vol, vmask=vmask, lat=lat); os.replace(tmp, out)   # never leave half-written files
        return uid, int(vmask.sum())
    except Exception as e:
        print("FAILED", uid, repr(e)[:200]); return uid, 0

def find_cache_files():
    """uid -> .npz path, from this session's cache and from any attached earlier output with the SAME preprocessing."""
    found = {}
    roots = [p for p in (LOCAL_CACHE, WORK, INPUT_ROOT) if p.exists()]
    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
            base = os.path.basename(dirpath)
            if base.startswith("knee_cache_s") and base.endswith(CACHE_TAG):        # same preprocessing only
                for fn in filenames:
                    if fn.endswith(".npz") and not fn.endswith(".tmp.npz"):
                        found.setdefault(fn[:-4], os.path.join(dirpath, fn))
    return found

def build_cache(uids, out_dir):
    global _SERIES, _OUT_DIR
    import shutil
    out_dir.mkdir(parents=True, exist_ok=True); _OUT_DIR = out_dir
    free = shutil.disk_usage(out_dir).free / 1e9
    need = len(uids) * NV * DEPTH * IMG * IMG * 0.5 / 1e9
    print(f"caching {len(uids)} studies -> {out_dir}  (free disk {free:.0f} GB; needs ~{need:.0f} GB)")
    if need > 0.9 * free:
        print("WARNING: the cache may not fit on this disk. Build it in MODE='cache' CPU sessions (SHARD=0..3) and attach them.")
    trs = pd.read_csv(COMP_DIR / "train_series.csv")
    trs["StudyInstanceUID"] = trs.StudyInstanceUID.astype(str); trs["SeriesInstanceUID"] = trs.SeriesInstanceUID.astype(str)
    _SERIES = {k: g for k, g in trs.groupby("StudyInstanceUID")}
    t = time.time(); rows = []
    with Pool(CACHE_PROCS) as pool:
        for uid, nv in tqdm(pool.imap_unordered(_cache_one, uids, chunksize=2), total=len(uids), mininterval=30):
            rows.append((uid, nv))
    man = pd.DataFrame(rows, columns=["StudyInstanceUID", "n_views"])
    man.to_csv(out_dir / "manifest.csv", index=False)
    print(f"cached: {int((man.n_views != 0).sum())}/{len(man)} ok ({int((man.n_views == -1).sum())} already there), "
          f"{(time.time()-t)/60:.1f} min")
    subprocess.run(["du", "-sh", str(out_dir)])

if MODE == "full":
    all_uids = sorted(pd.read_csv(COMP_DIR / "train.csv").StudyInstanceUID.astype(str))
    have = find_cache_files()
    todo = [u for u in all_uids if u not in have]
    print(f"cache [{CACHE_TAG}]: {len(all_uids) - len(todo)}/{len(all_uids)} studies already available "
          f"(this session or attached output)")
    if todo: build_cache(todo, LOCAL_CACHE)
    else: print("nothing to cache: skipping straight to training")
elif MODE == "cache":
    build_cache(sorted(pd.read_csv(COMP_DIR / "train.csv").StudyInstanceUID.astype(str))[SHARD::N_SHARDS],
                WORK / f"knee_cache_s{SHARD}_{CACHE_TAG}")

## 4. Report labels: your multilingual regex labeler (v5, unchanged)
Kept as one of the label sources. Section 5 weights it against the LLM-extracted sources per finding.

In [ ]:
def norm_text(t):
    t = str(t).lower().replace("ı", "i").replace("ß", "ss")
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"non[- ]?displaced", "undisplaced", t)
    return t

def split_sents(t):
    parts = re.split(r"[\r\n]+|(?<!\d)\.(?!\d)|;|•|。|\u2022", t)
    return [re.sub(r"\s+", " ", p).strip() for p in parts if p and p.strip()]

def R(*alts):  # word-start anchored alternation
    return re.compile(r"(?<![\w])(?:" + "|".join(alts) + r")")

CLAUSE = re.compile(r",|\bbut\b|\bhowever\b|\bpero\b|\bmais\b|\baber\b|\bama\b|\bmaar\b|\bmentre\b|\bwhereas\b")
NEG_PRE_BASE = [r"no", r"not", r"without", r"negative for", r"free of", r"absence of", r"absent", r"rule out", r"r/o",
                r"sin", r"ausencia", r"descarta\w*", r"sem", r"nao", r"pas de", r"pas d", r"sans", r"aucun\w*", r"absence",
                r"kein\w*", r"ohne", r"nicht", r"geen", r"zonder", r"niet", r"senza", r"non", r"assenza", r"nessun\w*",
                r"nema", r"bez", r"ne", r"nije", r"nisu", r"nigdje", r"без", r"няма", r"не", r"нито", r"δεν", r"χωρις", r"ουτε", r"ουδεν", r"μη"]
NEG_PRE    = re.compile(r"\b(?:" + "|".join(NEG_PRE_BASE) + r")\b")
NEG_PRE_PT = re.compile(r"\b(?:" + "|".join(x for x in NEG_PRE_BASE if x != "no") + r")\b")  # 'no' = 'in the' in PT
NEG_POST = re.compile(r"\b(?:yok\w*|izlenme\w*|saptanma\w*|gozlenme\w*|gorulme\w*|mevcut degil\w*|"
                      r"not (?:seen|identified|present|demonstrated)|excluded|ausgeschlossen|"
                      r"nicht (?:nachweisbar|abgrenzbar|erkennbar)|non (?:visibile|evidente)|no se (?:observa|identifica|evidencia)|"
                      r"nao (?:se )?(?:observa|identifica)|intact\w*|integr[oa]|intakt\w*|normal\w*|preserv\w*|conservad\w*|unremarkable|"
                      r"regelrecht\w*|unauffallig\w*|olagan\w*|uredn\w*|odrzan\w*|ocuvan\w*|b\.?o\b|без особености|нормал\w*|запазен\w*|"
                      r"φυσιολογ\w*|ακεραι\w*|χωρις)\b")
UNDERSTOOD = re.compile(r"anomal|normal|hoffa|baker|quiste|contus|rodilla|knee|knie|genou|ginocchio|diz\b|koljen|колян|γονατ|menisc|menisk|menisque|ligament|kreuzband|cartilag|knorpel|kikirdak|effusion|derrame|erguss|"
                        r"epanchement|efuzyon|versamento|fractur|fraktur|frattur|kirik|marrow|oseo|osseux|knochen|kemik|"
                        r"osso|kost|менискус|лигамент|μηνισκ|συνδεσμ|χιαστ|patell|rotul|condyl|condil|kondyl|capraz")

# ---------- structures ----------
MED_MEN = R(r"(?:medial|interno|interne|internal|mediale?[nsr]?|inner|ic|medijaln\w*|медиал\w*|εσω)\W+(?:\w+\W+){0,2}?(?:menisc|menisk|menisque|μηνισκ|менискус)",
            r"(?:menisc|menisk|menisque)\w*\W+(?:\w+\W+){0,2}?(?:medial|interno|interne|mediale|medialis)",
            r"innenmeniskus", r"both menisci", r"ambos meniscos", r"beide menisk\w*", r"les deux menisques", r"her iki menisk")
LAT_MEN = R(r"(?:lateral|externo|externe|external|laterale?[nsr]?|outer|dis|lateraln\w*|латерал\w*|εξω)\W+(?:\w+\W+){0,2}?(?:menisc|menisk|menisque|μηνισκ|менискус)",
            r"(?:menisc|menisk|menisque)\w*\W+(?:\w+\W+){0,2}?(?:lateral|externo|externe|laterale|lateralis)",
            r"aussenmeniskus", r"both menisci", r"ambos meniscos", r"beide menisk\w*", r"les deux menisques", r"her iki menisk",
            r"medial and lateral menisc", r"medial y lateral")
ACL_S = R(r"acl\b", r"lca\b", r"anterior cruciate", r"cruciate ligament,? anterior", r"cruzado anterior", r"croise anterieur",
          r"vordere[sn]? kreuzband", r"vkb\b", r"voorste kruisband", r"on capraz", r"ocb\b", r"crociato anteriore",
          r"prednj\w* kri\w*", r"предн\w* кръстн\w*", r"voorste kruisband\w*", r"προσθ\w* χιαστ\w*")
MCL_S = R(r"mcl\b", r"medial collateral", r"tibial collateral", r"colateral (?:medial|tibial)", r"collateral(?:e)? medial\w*",
          r"lli\b", r"lateral interne", r"lcm\b", r"innenband", r"mediale[sn]? (?:kollateral|seiten)band", r"mediaal collateraal",
          r"ic yan bag", r"mediale collaterale", r"medial kollateral\w*", r"collaterale mediale", r"medijaln\w* kolateral\w*", r"медиал\w* колатерал\w*", r"εσω πλαγι\w*")
TRI = r"tricompart\w*|three compartments|tres compartimentos|trikompart\w*|dreikompart\w*|pancompart\w*|all (?:three )?compartments"
MED_COMP = R(r"medial\w*\W+(?:\w+\W+){0,2}?(?:compart\w*|kompart\w*|femoro-?tibial\w*|tibio-?femoral\w*|condyl\w*|kondyl\w*|kondil\w*|condil\w*|kodnil\w*|plateau|platillo|meseta|plato|gelenk\w*|joint|femur\w*|tibi\w*)",
             r"(?:compart\w*|femoro-?tibial\w*|tibio-?femoral\w*|condil\w*|condyl\w*|kondyl\w*|plateau|platillo|meseta|plato)\W+(?:\w+\W+){0,2}?(?:medial|interne?|interno|mediale)",
             r"innen\w*", r"ic (?:kompartman|femorotibial)", r"medijaln\w*", r"медиал\w*", r"εσω", TRI)
LAT_COMP = R(r"lateral\w*\W+(?:\w+\W+){0,2}?(?:compart\w*|kompart\w*|femoro-?tibial\w*|tibio-?femoral\w*|condyl\w*|kondyl\w*|kondil\w*|condil\w*|kodnil\w*|plateau|platillo|meseta|plato|gelenk\w*|joint|femur\w*|tibi\w*)",
             r"(?:compart\w*|femoro-?tibial\w*|tibio-?femoral\w*|condil\w*|condyl\w*|kondyl\w*|plateau|platillo|meseta|plato)\W+(?:\w+\W+){0,2}?(?:lateral|externe?|externo|laterale)",
             r"aussen\w*", r"dis (?:kompartman|femorotibial)", r"lateraln\w*", r"латерал\w*", r"εξω", TRI)
PF_COMP = R(r"patell\w*", r"patel\w*", r"rotul\w*", r"trochle\w*", r"troclea\w*", r"trokle\w*", r"femoro-?patel\w*",
            r"retropatell\w*", r"femoro-?rotul\w*", r"пател\w*", r"επιγονατ\w*", TRI)

# ---------- findings / modifiers ----------
LIG_TEAR = R(r"tear\w*", r"torn", r"ruptur\w*", r"rotura", r"desgarr\w*", r"dechirure\w*", r"\w*riss\w*", r"scheur\w*", r"yirti\w*",
             r"kopu\w*", r"rottura", r"lacera\w*", r"discontinu\w*", r"disrupt\w*", r"avuls\w*", r"pukn\w*", r"разкъс\w*",
             r"руптур\w*", r"ρηξ\w*", r"прекъсван\w*", r"lesion complet\w*", r"lesao complet\w*", r"high[- ]grade", r"grade (?:3|iii)\b", r"grado (?:3|iii)\b")
LIG_BOOST = R(r"high[- ]grade", r"complete\w*", r"full[- ]thickness", r"total\w*", r"complet\w*", r"komplett\w*", r"tam\b",
              r"potpun\w*", r"пълн\w*", r"πληρ\w*", r"ολικ\w*", r"υφολικ\w*", r"volledig\w*", r"subtotal\w*", r"discontinu\w*", r"grade (?:3|iii)\b", r"grado (?:3|iii)\b")
ACL_EXCL = R(r"low[- ]grade", r"sprain\w*", r"esguince", r"entorse", r"distension", r"zerrung", r"elongat\w*", r"mucoid\w*",
             r"mukoid\w*", r"degenerat\w*", r"thicken\w*", r"engros\w*", r"epaissi\w*", r"verdick\w*", r"kalinlas\w*",
             r"intrasubstan\w*", r"miksoid\w*", r"dejener\w*", r"partial\w*", r"parcial\w*", r"partiel\w*", r"kismi", r"parziale", r"djelomi\w*",
             r"частичн\w*", r"μερικ\w*", r"reconstru\w*", r"plast\w*", r"graft\w*", r"injert\w*", r"greffe", r"greft\w*", r"trapianto")
ACL_OVERRIDE = re.compile(r"not (?:well )?visuali[sz]ed|non[- ]?visuali[sz]\w*|no se visualiza|nao (?:se )?visualiza|nicht (?:mehr )?(?:abgrenzbar|darstellbar)|non visualise")
MCL_WEAK = R(r"sprain\w*", r"esguince", r"entorse", r"distorsion\w*", r"zerrung", r"burk\w*", r"distorsione")
MCL_EXCL = R(r"low[- ]grade", r"grade (?:1|i)\b", r"grado (?:1|i)\b", r"grad (?:1|i)\b", r"graad (?:1|i)\b", r"chronic\w*", r"cronic\w*",
             r"chroni\w*", r"kronik", r"remote", r"old\b", r"antigu\w*", r"ancien\w*", r"eski", r"vecchi\w*", r"thicken\w*", r"engros\w*",
             r"epaissi\w*", r"verdick\w*", r"verdikt", r"pellegrini", r"mild")
MCL_MID = R(r"partial\w*", r"parcial\w*", r"partiel\w*", r"parti ele", r"kismi", r"parziale", r"djelomi\w*", r"частичн\w*", r"μερικ\w*",
            r"grade (?:2|ii)\b", r"grado (?:2|ii)\b", r"graad (?:2|ii)\b", r"grad (?:2|ii)\b")
MCL_INJ = R(r"injur\w*", r"lesion\w*", r"lesao", r"verletz\w*", r"letsel\w*", r"hasar\w*", r"ozljed\w*", r"увред\w*", r"κακωσ\w*")
MEN_TEAR = R(r"tear\w*", r"torn", r"rotura", r"desgarr\w*", r"ruptur\w*", r"dechirure\w*", r"fissur\w*", r"\w*riss\w*", r"scheur\w*",
             r"yirti\w*", r"rottura", r"lacera\w*", r"lesion\w*", r"lesao", r"lasion\w*", r"lesione", r"pukn\w*", r"bucket",
             r"asa de balde", r"anse de seau", r"korbhenkel\w*", r"flap", r"colgajo", r"radial", r"obliqu\w*", r"horizontal",
             r"longitudinal", r"complex", r"truncat\w*", r"blunt\w*", r"displac\w*", r"desplaz\w*", r"luxat\w*",
             r"grade (?:3|iii)\b", r"grado (?:3|iii)\b", r"grad (?:3|iii)\b", r"stade (?:3|iii)\b", r"tip (?:3|iii)\b",
             r"разкъс\w*", r"руптур\w*", r"ρηξ\w*", r"lezij\w*", r"rascjep\w*", r"увред\w*", r"лези\w*", r"amputa\w*", r"scheur\w*")
MEN_BOOST = R(r"tear\w*", r"torn", r"rotura", r"desgarr\w*", r"dechirure\w*", r"\w*riss\w*", r"yirti\w*", r"rottura", r"ruptur\w*",
              r"bucket", r"flap", r"displac\w*", r"truncat\w*", r"grade (?:3|iii)\b", r"grado (?:3|iii)\b", r"(?:grade|степен|стади\w*)\s*[–-]?\s*(?:3|iii)[abcабв]?\b", r"(?<!\d)3[abcабв]?\s*(?:степен|стади\w*|grade)", r"[–-]\s*3[abcабв]?\b")
MEN_EXCL = R(r"intrasubstan\w*", r"degenerat\w*", r"mucoid\w*", r"mukoid\w*", r"myxoid\w*", r"mixoid\w*",
             r"grade (?:1|2|i|ii)\b", r"grado (?:1|2|i|ii)\b", r"grad (?:1|2|i|ii)\b", r"stade (?:1|2|i|ii)\b", r"tip (?:1|2|i|ii)\b",
             r"does not (?:reach|extend)", r"not (?:reaching|extending)", r"no (?:alcanza|llega)", r"nao atinge", r"n.atteint pas",
             r"yuzeye ulasma\w*", r"non raggiunge", r"(?:grade|степен|стади\w*|тип|stupanj|stupnj\w*)\s*[–-]?\s*(?:1|2|i|ii)[abcабв]?\b", r"(?<!\d)(?:1|2)[abcабв]?\s*(?:степен|стади\w*|grade|stupnj\w*)", r"[–-]\s*(?:1|2)[abcабв]\b", r"не достига\w*", r"dejener\w*", r"meniskopati\w*", r"miksoid\w*", r"тип (?:1|2)\w*", r"степен (?:1|2|i|ii)\b", r"stupnj\w* (?:1|2|i|ii)\b", r"(?:1|2|i|ii)\.? stupnj\w*", r"εκφυλ\w*", r"post-?op\w*", r"resect\w*", r"men[i]?s?cectom\w*", r"meniscectom\w*")
OA_WORDS = R(r"(?:osteo)?arthr\w*", r"artro\w*", r"artros\w*", r"gonart\w*", r"osteoartr\w*", r"chondr\w*", r"condr\w*",
             r"cartilag\w*", r"knorpel\w*", r"kikirdak\w*", r"hrskavi\w*", r"хрущял\w*", r"χονδρ\w*", r"kondro\w*", r"hondr\w*", r"хондр\w*", r"артроз\w*", r"остеоартр\w*", r"αρθρ\w*", r"kraakbeen\w*", r"gonarthr\w*", r"kondral\w*", r"kondromalaz\w*")
OA_HIGH = R(r"full[- ]thickness", r"high[- ]grade", r"grade (?:3|4|iii|iv)\b", r"grado (?:3|4|iii|iv)\b", r"grad (?:3|4|iii|iv)\b",
            r"stade (?:3|4|iii|iv)\b", r"evre (?:3|4)\b", r"outerbridge (?:3|4|iii|iv)\b", r"kl (?:3|4)\b", r"severe\w*", r"sever[oa]",
            r"schwer\w*", r"hochgradig\w*", r"ausgepragt\w*", r"fortgeschritten\w*", r"ileri", r"ciddi", r"grave", r"avanzad\w*",
            r"avancad\w*", r"avance\w*", r"advanced", r"marked\w*", r"bone[- ]on[- ]bone", r"denud\w*", r"exposed", r"expuest\w*",
            r"eburn\w*", r"espesor (?:completo|total)", r"espessura (?:total|completa)", r"pleine epaisseur", r"transfix\w*",
            r"vollschicht\w*", r"tam kat", r"tutto spessore", r"complete loss", r"near[- ]complete", r"teska", r"тежк\w*", r"σοβαρ\w*",
            r"(?:3|4|iii|iv)\.? stupnj\w*", r"stupnj\w* (?:3|4|iii|iv)\b", r"pune debljine", r"potpun\w* gubit\w*", r"icrs (?:grade )?(?:3|4|iii|iv)\b",
            r"степен (?:3|4|iii|iv)\b", r"(?:3|4|iii|iv) степен", r"пълн\w* дебелин\w*", r"изразен\w*", r"πληρ\w* (?:εξαλειψ\w*|παχ\w*|απωλει\w*)", r"βαθμου (?:3|4|iii|iv)",
            r"gevorderd\w*", r"volledig\w*", r"ernstig\w*", r"uitgesproken")
OA_MID = R(r"moderat\w*", r"moderad\w*", r"modere\w*", r"massig\w*", r"orta", r"umjeren\w*", r"умерен\w*", r"μετρι\w*", r"matig\w*")
OA_MILD = R(r"mild\w*", r"minimal", r"minor", r"early", r"incipient\w*", r"inicial", r"leve", r"legere?", r"leicht\w*", r"gering\w*",
            r"beginnend\w*", r"hafif", r"erken", r"lieve", r"iniziale", r"discret\w*", r"superficial\w*", r"superficiel\w*",
            r"oberflachlich\w*", r"yuzeyel", r"grade (?:1|2|i|ii)\b", r"grado (?:1|2|i|ii)\b", r"grad (?:1|2|i|ii)\b",
            r"stade (?:1|2|i|ii)\b", r"partial[- ]thickness", r"espesor parcial", r"espessura parcial", r"epaisseur partielle", r"teilschicht\w*", r"kismi kat", r"djelomi\w*", r"ηπι\w*", r"леки\w*", r"лек\b", r"beginnend\w*", r"gering\w*", r"small", r"pequen\w*", r"petit\w*", r"klein\w*", r"kucuk", r"piccol\w*", r"slight\w*", r"softening")
BIG = R(r"moderat\w*", r"moderad\w*", r"modere\w*", r"large\w*", r"abundant\w*", r"abondant\w*", r"importante?", r"marked",
        r"significant\w*", r"massive", r"voluminous", r"volumin\w*", r"grande?", r"gross\w*", r"ausgepragt\w*", r"deutlich\w*",
        r"reichlich\w*", r"kraftig\w*", r"massig\w*", r"belirgin", r"fazla", r"yogun", r"orta", r"abbondant\w*", r"cospicu\w*",
        r"umjeren\w*", r"obil\w*", r"velik\w*", r"значителн\w*", r"умерен\w*", r"μετρι\w*", r"μεγαλ\w*", r"σημαντικ\w*",
        r"distend\w*", r"distens\w*", r"tense", r"opsez\w*", r"obiln\w*", r"vec\w*", r"изразен\w*", r"голям\w*", r"значим\w*", r"εκτεταμ\w*", r"εκταταμ\w*", r"ευμεγεθ\w*", r"uitgebreid\w*", r"ruim\w*", r"opzetting")
SMALL = R(r"small", r"trace", r"tiny", r"minimal", r"minim\w*", r"mild", r"slight\w*", r"physiolog\w*", r"fisiolog\w*", r"pequen\w*",
          r"escas\w*", r"discret\w*", r"leve", r"ligero", r"poco", r"pouco", r"petit\w*", r"faible", r"minime", r"klein\w*",
          r"gering\w*", r"wenig\w*", r"leicht\w*", r"az\b", r"hafif", r"kucuk", r"piccol\w*", r"lieve", r"scarso", r"modest\w*",
          r"mal[aio]\b", r"malen\w*", r"manj\w*", r"minimaln\w*", r"diskretn\w*", r"oskudn\w*", r"blag\w*", r"fiziolosk\w*", r"малк\w*", r"малък", r"минимал\w*", r"незначим\w*", r"ηπι\w*", r"beperkt\w*", r"gering\w*", r"weinig", r"незначит\w*", r"μικρ\w*", r"ελαχ\w*")
EFF = R(r"effusion\w*", r"derrame\w*", r"epanchement\w*", r"\w*erguss\w*", r"efuzyon\w*", r"sivi", r"versamento\w*", r"hydrops",
        r"hidrops", r"hydrarthr\w*", r"hidrartr\w*", r"\w*vocht\w*", r"izljev\w*", r"излив\w*", r"συλλογ\w*", r"υγρου ενδαρθρ\w*", r"gewrichtsvocht", r"hydrops", r"joint fluid",
        r"intra-?articular fluid", r"liquido (?:articular|intra-?articular)", r"liquide (?:articulaire|intra-?articulaire)")
BAKER = R(r"baker\w*", r"popliteal cyst", r"cyst\w* (?:in the )?poplit\w*", r"quiste popliteo", r"cisto popliteo", r"kyste poplite\w*",
          r"poplitealzyste", r"popliteal kist", r"cisti poplitea", r"gastrocnemi\w*[- ]semimembran\w*", r"semimembran\w*[- ]gastrocnemi\w*",
          r"poplitealn\w* cist\w*", r"бейкер\w*", r"бекер\w*", r"поплите\w* кист\w*", r"μπεικερ", r"ιγνυακ\w* κυστ\w*", r"popliteale cyste")
SYN = R(r"synovit\w*", r"sinovit\w*", r"synovialit\w*", r"synovial (?:thicken\w*|hypertroph\w*|proliferat\w*|inflamm\w*|enhancement)",
        r"(?:engrosamiento|hipertrofia|proliferacion) sinovial", r"espessamento sinovial", r"(?:epaississement|hypertrophie) synovial\w*",
        r"synovialverdick\w*", r"synoviale verdick\w*", r"sinovyal (?:kalinlas\w*|hipertrofi\w*)", r"(?:ispessimento|ipertrofia) sinovial\w*",
        r"sinovijal\w*", r"синовит\w*", r"υμενιτ\w*", r"synovi\w* (?:verdik\w*|zwelling)", r"verdikking\w* van (?:het )?synovium", r"thicken\w* synovi\w*", r"synovial tissue")
SYN_MILD = R(r"mild\w*", r"minimal", r"leve", r"legere?", r"leicht\w*", r"hafif", r"lieve", r"discret\w*", r"slight\w*")
CONT = R(r"contus\w*", r"bruis\w*", r"(?:bone )?marrow (?:o)?edema\w*", r"(?:o)?edema (?:of the )?(?:bone )?marrow",
         r"edema (?:oseo|osea|medular|de (?:la )?medula|oss\w*|da medula|midollare|spongios\w*)", r"(?:o)?edeme (?:osseux|medullaire|spongieux|de la moelle)",
         r"knochenmarks?odem\w*", r"bone[- ]bruise\w*", r"kemik ili\w* odem\w*", r"kemik ili\w*", r"kontuz\w*", r"edem\w* kost\w*", r"kostan\w* edem\w*", r"kosti edem\w*",
         r"костен\w* оток", r"костно[- ]?мозъч\w* едем", r"контузи\w*", r"οιδημα\w*(?: \w+){0,3} μυελ\w*", r"οστεομυελικ\w* οιδημ\w*", r"θλαση",
         r"beenmergoedeem\w*", r"(?:been)?merg ?oedeem\w*", r"botkneuzing\w*", r"bone bruise")
CONT_BOOST = R(r"contus\w*", r"bruis\w*", r"kontuz\w*", r"контузи\w*", r"θλαση", r"trauma\w*", r"impact\w*", r"pivot\w*", r"kissing", r"контузион\w*", r"kontuzij\w*", r"posttraum\w*", r"traum\w*")
CONT_EXCL = R(r"subchondral\w*", r"subcondral\w*", r"sous-?chondral\w*", r"degenerat\w*", r"reactiv\w*", r"reaktiv\w*", r"stress",
              r"insufficien\w*", r"insuficien\w*", r"osteoarthr\w*", r"artros\w*", r"arthros\w*", r"geod\w*", r"cyst\w*", r"kist\w*", r"kyst\w*", r"субхондрал\w*", r"υποχονδρ\w*", r"subhondral\w*", r"εκφυλ\w*", r"degenerativ\w*")
FRAC = R(r"fractur\w*", r"fratur\w*", r"frattur\w*", r"fraktur\w*", r"kirik\w*", r"kirig\w*", r"breuk\w*", r"prijelom\w*",
         r"фрактур\w*", r"счупван\w*", r"καταγμ\w*", r"infrakcij\w*", r"infraktion\w*", r"fractuur\w*", r"segond", r"avuls\w*", r"impaction", r"osteochondral (?:fracture|fragment)")
FRAC_BOOST = R(r"acute", r"agud\w*", r"recent\w*", r"akut\w*", r"frais", r"fresh", r"yeni", r"свеж\w*", r"οξε\w*")
FRAC_EXCL = R(r"old\b", r"heal\w*", r"remote", r"chronic\w*", r"cronic\w*", r"chroni\w*", r"kronik", r"consolid\w*", r"ancien\w*",
              r"alte[n]?\b", r"verheilt\w*", r"eski", r"pregress\w*", r"vecchi\w*",
              r"sequel\w*", r"secuel\w*", r"стар\w*", r"παλαι\w*", r"patholog\w*", r"patolog\w*")

# spec: struct, find, base, boost(->1), mid(->mid_score), excl(->0), weak(regex, weak_score), override(->1)
SPEC = {
    "ACL":              dict(struct=ACL_S, find=LIG_TEAR, base=1.0, boost=LIG_BOOST, excl=ACL_EXCL, override=ACL_OVERRIDE),
    "MCL":              dict(struct=MCL_S, find=LIG_TEAR, base=1.0, boost=LIG_BOOST, mid=(MCL_MID, 0.7), excl=MCL_EXCL, weak=(MCL_WEAK, 0.3)),
    "MCL_INJ":          dict(struct=MCL_S, find=MCL_INJ, base=0.3, boost=LIG_BOOST, mid=(MCL_MID, 0.6), excl=MCL_EXCL, target="MCL"),
    "Medial Meniscus":  dict(struct=MED_MEN, find=MEN_TEAR, base=0.6, boost=MEN_BOOST, excl=MEN_EXCL),
    "Lateral Meniscus": dict(struct=LAT_MEN, find=MEN_TEAR, base=0.6, boost=MEN_BOOST, excl=MEN_EXCL),
    "Medial OA":        dict(struct=MED_COMP, find=OA_WORDS, base=0.3, boost=OA_HIGH, mid=(OA_MID, 0.6), excl=OA_MILD),
    "Lateral OA":       dict(struct=LAT_COMP, find=OA_WORDS, base=0.3, boost=OA_HIGH, mid=(OA_MID, 0.6), excl=OA_MILD),
    "PF OA":            dict(struct=PF_COMP, find=OA_WORDS, base=0.3, boost=OA_HIGH, mid=(OA_MID, 0.6), excl=OA_MILD),
    "Effusion":         dict(struct=None, find=EFF, base=0.6, boost=BIG, mid=(SMALL, 0.3)),
    "Synovitis":        dict(struct=None, find=SYN, base=1.0, mid=(SYN_MILD, 0.5)),
    "Baker's":          dict(struct=None, find=BAKER, base=0.6, boost=BIG, mid=(SMALL, 0.25)),
    "Contusion":        dict(struct=None, find=CONT, base=0.7, boost=CONT_BOOST, excl=CONT_EXCL),
    "Fracture":         dict(struct=None, find=FRAC, base=1.0, boost=FRAC_BOOST, excl=FRAC_EXCL),
}

def _score_match(sp, s, m, neg_pre):
    pre = CLAUSE.split(s[:m.start()])[-1]
    post = CLAUSE.split(s[m.end():])[0]
    if neg_pre.search(pre) or NEG_POST.search(post[:40]):
        return None
    clause = pre + " " + m.group(0) + " " + post[:120]
    if sp.get("boost") is not None and sp["boost"].search(clause): return 1.0
    if sp.get("mid") is not None and sp["mid"][0].search(clause): return sp["mid"][1]
    if sp.get("excl") is not None and sp["excl"].search(clause): return 0.0
    return sp["base"]

def label_report(report):
    t = norm_text(report if isinstance(report, str) else "")
    is_pt = bool(re.search(r"\b(?:nao|sem|joelho|presenca|nota-se|observa-se|\w+coes?|\w+cao)\b", t))
    neg_pre = NEG_PRE_PT if is_pt else NEG_PRE
    scores = {l: 0.0 for l in LABELS}
    for s in split_sents(t):
        for lab, sp in SPEC.items():
            if sp["struct"] is not None and not sp["struct"].search(s):
                continue
            if sp.get("override") is not None and sp["override"].search(s):
                scores[lab] = 1.0; continue
            for m in sp["find"].finditer(s):
                v = _score_match(sp, s, m, neg_pre)
                if v is not None: scores[sp.get("target", lab)] = max(scores[sp.get("target", lab)], v)
            if sp.get("weak") is not None:
                for m in sp["weak"][0].finditer(s):
                    v = _score_match(dict(sp, base=sp["weak"][1]), s, m, neg_pre)
                    if v is not None: scores[sp.get("target", lab)] = max(scores[sp.get("target", lab)], v)
    understood = bool(UNDERSTOOD.search(t))
    return scores, understood
print('regex labeler ready')

## 5. Label fusion: LLM label datasets + regex, weighted per finding by agreement with the 58 gold studies
Any attached CSV that has a study-id column and ≥ 8 of the 12 findings is picked up automatically
(e.g. `pilkwang/rsna-knee-llm-labels`). Each source gets, per finding, weight `(AUC_vs_gold − 0.5)²`, so a source
that is unreliable on Synovitis has little say on Synovitis only.
The gold AUC of the *fused* labels is mildly optimistic (weights are fit on the same 58 studies), so treat it as a guide.

**v7 leak guard.** A table whose values equal the gold labels on ≥ 90% of the 58 gold studies' cells has
copied them, so its gold AUC means nothing. It keeps a per-finding weight equal to the mean of the clean sources
(it may still be a good labeler on the other 4,349 studies) and the FUSED gold AUC is then marked as not trustworthy.

In [ ]:
from sklearn.metrics import roc_auc_score
def _norm(s): return re.sub(r"[^a-z]", "", str(s).lower())
_LAB_KEYS = {_norm(l): l for l in LABELS}
_LAB_KEYS.update({"bakerscyst": "Baker's", "bakercyst": "Baker's", "baker": "Baker's", "popliteal": "Baker's",
                  "patellofemoraloa": "PF OA", "patellofemoral": "PF OA", "pfoa": "PF OA",
                  "medialmeniscustear": "Medial Meniscus", "lateralmeniscustear": "Lateral Meniscus",
                  "acltear": "ACL", "mcltear": "MCL", "jointeffusion": "Effusion", "bonecontusion": "Contusion",
                  "bonebruise": "Contusion", "medialtfoa": "Medial OA", "lateraltfoa": "Lateral OA"})
_STRIP = re.compile(r"^(llm|label|labels|prob|pred|p|y|target|soft)+|(llm|label|prob|pred|p|y|target|soft|score)+$")

def map_columns(cols):
    m = {}
    for c in cols:
        n = _norm(c)
        lab = _LAB_KEYS.get(n) or _LAB_KEYS.get(_STRIP.sub("", n))
        if lab and lab not in m.values(): m[c] = lab
    return m

def find_label_sources():
    srcs = []
    for dirpath, _, filenames in walk_inputs():
        for fn in filenames:
            if not fn.lower().endswith(".csv") or fn.startswith(("oof_", "sample_submission", "submission", "manifest",
                                                                  "train", "test", "accuracy", "stack", "goldpreds")):
                continue
            p = Path(dirpath) / fn
            try: head = pd.read_csv(p, nrows=5)
            except Exception: continue
            idc = [c for c in head.columns if "study" in c.lower()]
            m = map_columns(head.columns)
            if idc and len(m) >= 8:
                df = pd.read_csv(p)
                df = df.rename(columns={idc[0]: "StudyInstanceUID", **m})
                df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
                vals = df[list(m.values())].apply(pd.to_numeric, errors="coerce")
                if vals.max().max() > 1.5: vals = vals / 100.0
                df[list(m.values())] = vals.clip(0, 1)
                df = df.groupby("StudyInstanceUID")[list(m.values())].mean()
                srcs.append((str(p.relative_to(INPUT_ROOT)), df.reindex(columns=LABELS)))
    return srcs

def gold_auc(g, p):
    ok = ~np.isnan(g) & ~np.isnan(p)
    g, p = g[ok], p[ok]
    if len(g) < 5 or g.min() == g.max(): return np.nan
    return roc_auc_score((g >= 0.5).astype(int), p)

def build_targets(tr):
    """-> fused soft labels [N,12], label mask [N,12], gold labels df, report table."""
    uids = tr.StudyInstanceUID.astype(str).values
    gold = tr[LABELS].apply(pd.to_numeric, errors="coerce").values
    is_gold = ~np.isnan(gold).all(1)
    res = tr["Report"].map(label_report)
    regex = np.array([[r[0][l] for l in LABELS] for r in res], np.float32)
    sources = [("regex_v5", regex)]
    for name, df in find_label_sources():
        sources.append((name, df.reindex(uids).values.astype(np.float32)))
    print("label sources:", [s[0] for s in sources])
    table, W, leaky = [], np.zeros((len(sources), len(LABELS))), []
    for si, (name, arr) in enumerate(sources):
        aucs = [gold_auc(gold[is_gold, j], arr[is_gold, j]) for j in range(len(LABELS))]
        W[si] = [max(a - 0.5, 0.02) ** 2 if not np.isnan(a) else 0.0 for a in aucs]
        g, a = gold[is_gold], arr[is_gold]
        ok = ~np.isnan(g) & ~np.isnan(a)
        copied = float((np.abs(g[ok] - a[ok]) < 1e-6).mean()) if ok.sum() >= 50 else 0.0
        if si > 0 and copied >= 0.9:
            leaky.append(si); name = name + "  [LEAK: copies gold labels]"
        table.append([name] + aucs + [np.nanmean(aucs)])
    clean = [si for si in range(len(sources)) if si not in leaky]
    for si in leaky:                                             # average weight of the clean sources, per finding
        W[si] = W[clean].mean(0)
        print(f"!! {sources[si][0]} reproduces the gold labels on the 58 gold studies; its gold AUC is meaningless. "
              f"Using the clean sources' average weight for it.")
    num = np.zeros_like(regex); den = np.zeros_like(regex)
    for si, (_, arr) in enumerate(sources):
        have = ~np.isnan(arr)
        num += np.where(have, arr, 0) * W[si]; den += have * W[si]
    fused = np.where(den > 0, num / np.maximum(den, 1e-9), regex).astype(np.float32)
    fa = [gold_auc(gold[is_gold, j], fused[is_gold, j]) for j in range(len(LABELS))]
    table.append(["FUSED" + ("  [NOT trustworthy: a leaky source is in it]" if leaky else "")] + fa + [np.nanmean(fa)])
    rep = pd.DataFrame(table, columns=["source"] + LABELS + ["macro"]).set_index("source").round(3)
    LM = np.ones_like(fused)
    if len(sources) == 1:                                        # regex only: drop reports it could not parse
        und = np.array([r[1] for r in res]); LM[~und] = 0
    return fused, LM, gold, is_gold, rep

## 6. Model: shared 2.5D encoder → slice tokens → study transformer → one attention pool per finding
* Each sampled slice enters ConvNeXt as 3 channels (previous, current, next slice).
* Every slice token gets a view embedding and a continuous slice-position embedding, so the model knows *where* it is.
* A 2-layer transformer mixes the 60 tokens of the whole study (5 views × 12 slices), letting the coronal and sagittal
  evidence for the same structure talk to each other.
* One learned query per finding attends over the tokens: the ACL query can focus on 2–3 sagittal slices while the
  effusion query spreads over the suprapatellar pouch.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count() if DEVICE == "cuda" else 0
# One device per concurrently trained fold. No DataParallel: GPU-to-GPU communication is what freezes on Kaggle T4 x2.
_FAKE = int(os.environ.get("KNEE_FAKE_DEVICES", 0))            # testing only: simulate N devices on CPU
TRAIN_DEVICES = [f"cuda:{i}" for i in range(N_GPU)] if N_GPU else ["cpu"] * max(1, _FAKE)
if not PARALLEL_FOLDS: TRAIN_DEVICES = TRAIN_DEVICES[:1]
if DEVICE == "cuda": torch.backends.cudnn.benchmark = True
print("device:", DEVICE, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)],
      f"| folds in parallel: {len(TRAIN_DEVICES)} | batch {BATCH_SIZE} x accum {ACCUM} | grad-ckpt {GRAD_CKPT}",
      "| torch", torch.__version__, "| timm", timm.__version__)

def wrap(m):
    return (m, m)

if MODE in ("train", "full") and PRETRAINED and not ONLINE:
    raise RuntimeError("Training downloads pretrained weights: turn Internet ON (Settings -> Internet) for this run.")
if MODE in ("train", "infer", "full") and DEVICE == "cpu" and os.environ.get("KNEE_ALLOW_CPU") != "1":
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 (or P100), then restart the session.")

class KneeReader(nn.Module):
    def __init__(self, backbone=BACKBONE, pretrained=True, n_views=NV, n_out=len(LABELS)):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3, drop_path_rate=DROP_PATH)
        if GRAD_CKPT:
            try: self.enc.set_grad_checkpointing(True)
            except Exception: pass
        nf = self.enc.num_features
        self.proj = nn.Sequential(nn.Linear(nf, DIM), nn.LayerNorm(DIM))
        self.view_emb = nn.Parameter(torch.zeros(n_views, DIM)); nn.init.normal_(self.view_emb, std=0.02)
        self.pos = nn.Sequential(nn.Linear(1, 64), nn.GELU(), nn.Linear(64, DIM))
        self.lat = nn.Linear(3, DIM)
        layer = nn.TransformerEncoderLayer(DIM, HEADS, DIM * 4, dropout=0.1, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.mixer = nn.TransformerEncoder(layer, MIX_LAYERS, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(DIM)
        self.key = nn.Linear(DIM, DIM)
        self.queries = nn.Parameter(torch.randn(n_out, DIM) * 0.02)
        self.cls_w = nn.Parameter(torch.randn(n_out, DIM) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) * 255)
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1) * 255)

    def forward(self, x, z, vmask, lat):
        # x: [B, V, T, 3, H, W] (uint8 or float 0..255), z: [B, V, T] slice position 0..1, vmask: [B, V], lat: [B, 3]
        # autocast lives *inside* forward so it also applies in DataParallel replica threads
        B, V, T = x.shape[:3]
        pad = (vmask <= 0)[:, :, None].expand(B, V, T).reshape(B, V * T)
        pad = pad & ~pad.all(1, keepdim=True)                    # a study with no views: attend to zeros, not NaN
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=x.is_cuda):
            img = (x.flatten(0, 2).float() - self.mean) / self.std
            f = self.enc(img.contiguous(memory_format=torch.channels_last))
            f = self.proj(f).view(B, V, T, DIM)
            f = f + self.view_emb[None, :, None] + self.pos(z[..., None].float())
            tok = f.reshape(B, V * T, DIM) + self.lat(lat.float())[:, None]
            h = self.mixer(tok, src_key_padding_mask=pad)
        with torch.autocast(device_type="cuda", enabled=False):   # pooling + head in fp32
            h = self.norm(h.float())
            s = torch.einsum("bnd,ld->bln", self.key(h), self.queries) / math.sqrt(DIM)
            w = s.masked_fill(pad[:, None], -1e4).softmax(-1)
            pooled = torch.einsum("bln,bnd->bld", w, h)
            return (pooled * self.cls_w).sum(-1) + self.cls_b

def slice_idx(depth, t, mode, rng=None):
    """Indices of T slices out of DEPTH: jittered bins for training, two interleaved fixed sets for eval/TTA."""
    edges = np.linspace(0, depth, t + 1)
    if mode == "train":
        return np.array([rng.integers(int(edges[i]), max(int(edges[i]) + 1, int(edges[i + 1]))) for i in range(t)])
    off = 0.25 if mode == 0 else 0.75
    return np.clip((edges[:-1] + off * (edges[1:] - edges[:-1])).astype(int), 0, depth - 1)

def make_input(vol, idx):
    """vol [V, D, H, W] uint8 -> [V, T, 3, H, W] (prev, cur, next) and z [V, T]."""
    D = vol.shape[1]
    trip = np.stack([np.clip(idx - 1, 0, D - 1), idx, np.clip(idx + 1, 0, D - 1)], 1)    # [T, 3]
    x = vol[:, trip]                                                                    # [V, T, 3, H, W]
    z = np.broadcast_to((idx / max(D - 1, 1)).astype(np.float32), (vol.shape[0], len(idx))).copy()
    return np.ascontiguousarray(x), z

def gpu_augment(x):
    """Same random affine + intensity per view (across its slices). No flips: left/right is a label."""
    B, V, T, C, H, W = x.shape
    x = x.reshape(B * V, T * C, H, W).float()
    n, dev = x.shape[0], x.device
    ang = (torch.rand(n, device=dev) - 0.5) * 2 * math.radians(15)
    sc = 1 + (torch.rand(n, device=dev) - 0.5) * 0.25
    tx, ty = [(torch.rand(n, device=dev) - 0.5) * 0.16 for _ in range(2)]
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), align_corners=False)
    gamma = torch.exp((torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.6)
    x = 255 * (x / 255).clamp(0, 1) ** gamma
    x = x * (1 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.3) + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 30
    return x.clamp(0, 255).view(B, V, T, C, H, W)

class EMA:
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        self.n += 1; d = min(self.decay, (1 + self.n) / (10 + self.n))
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(d).add_(v.detach().float(), alpha=1 - d)
            else: self.shadow[k].copy_(v)
    def state_dict(self): return {k: v.clone() for k, v in self.shadow.items()}

def macro_auc(y, p, m=None):
    aucs = {}
    for j, lab in enumerate(LABELS):
        sel = np.ones(len(y), bool) if m is None else m[:, j] > 0
        yy = (y[sel, j] >= 0.5).astype(int)
        if len(yy) < 2 or yy.min() == yy.max(): continue
        aucs[lab] = roc_auc_score(yy, p[sel, j])
    return (float(np.mean(list(aucs.values()))) if aucs else float("nan")), aucs

@torch.no_grad()
def predict(models, loader, max_k=None):
    """Average sigmoid over models and over the TTA axis (loader yields x as [B, K, V, T, 3, H, W])."""
    if not isinstance(models, (list, tuple)): models = [models]
    for m in models: m.eval()
    out = []
    for b in loader:
        K = min(b[0].shape[1], max_k or b[0].shape[1]); acc = 0
        for m in models:
            d = next(m.parameters()).device
            x, z, vm, lat = [t.to(d, non_blocking=True) for t in b[:4]]
            for k in range(K):
                acc = acc + torch.sigmoid(m(x[:, k].contiguous(), z[:, k].contiguous(), vm, lat).float()).cpu()
        out.append((acc / (K * len(models))).numpy())
    return np.concatenate(out) if out else np.zeros((0, len(LABELS)), np.float32)

## 7. Training — all folds in `full` mode (or one fold per session in `train` mode)
* The 58 gold studies are **never** trained on (except in `FINAL_ALL_DATA`); they are the honest validation set.
* Model selection score = ½·gold macro AUC + ½·held-out weak macro AUC (gold alone is too noisy with ~10 positives per finding).
* Out-of-fold predictions are saved as `oof_v6_f{FOLD}.csv` for distillation and blend tuning.
* Time-aware: an epoch only starts if it can finish before the budget, and a new fold only starts if the previous
  fold's duration still fits. Each finished fold's weights are saved immediately, so a 1-, 2- or 3-fold run is still usable.

In [ ]:
import threading
_SEED_LOCK = threading.Lock()   # v12: members train in threads of one process

class CacheDS(Dataset):
    def __init__(self, uids, paths, Y, LM, Wt, train):
        self.uids, self.paths, self.Y, self.LM, self.Wt, self.train = list(uids), paths, Y, LM, Wt, train
    def __len__(self): return len(self.uids)
    def __getitem__(self, i):
        d = np.load(self.paths[self.uids[i]])
        vol, vm, lat = d["vol"], d["vmask"].astype(np.float32).copy(), d["lat"].astype(np.float32)
        if self.train:
            rng = np.random.default_rng()
            idx = slice_idx(vol.shape[1], T_TRAIN or T_SLICES, "train", rng)
            on = np.where(vm > 0)[0]
            if len(on) > 1:
                drop = on[rng.random(len(on)) < VIEW_DROP]
                if len(drop) < len(on): vm[drop] = 0
            x, z = make_input(vol, idx); x = x[None]; z = z[None]
        else:
            xs, zs = zip(*[make_input(vol, slice_idx(vol.shape[1], T_SLICES, 0))])
            x, z = np.stack(xs), np.stack(zs)
        x = x * (vm[None, :, None, None, None, None] > 0)
        return (torch.from_numpy(x), torch.from_numpy(z), torch.from_numpy(vm), torch.from_numpy(lat),
                torch.from_numpy(self.Y[i]), torch.from_numpy(self.LM[i]), torch.tensor(self.Wt[i]), torch.tensor(i))

from sklearn.model_selection import StratifiedKFold

def _refine_alpha(table):
    """Per-finding max share of the memory in the target: larger where the fused labels agree least with the gold studies."""
    base = np.full(len(LABELS), REFINE_ALPHA, np.float32)
    rows = [i for i in table.index if str(i).startswith("FUSED")]
    if not rows or "NOT trustworthy" in str(rows[0]): return base
    auc = np.nan_to_num(table.loc[rows[0], LABELS].astype(float).values, nan=0.85)
    return (REFINE_ALPHA * np.clip((0.97 - auc) / 0.15, REFINE_FLOOR, 1.0)).astype(np.float32)

def prepare_training():
    """Labels, cache paths and fold assignment, computed once and shared by every fold."""
    log("preparing labels (regex on 4,407 reports + label datasets)...")
    tr = pd.read_csv(COMP_DIR / "train.csv"); tr["StudyInstanceUID"] = tr.StudyInstanceUID.astype(str)
    Y, LM, gold, is_gold, table = build_targets(tr)
    print("label agreement with the 58 gold studies (AUC):"); print(table.to_string())
    table.to_csv(WORK / "label_sources_vs_gold.csv")
    if len(table) <= 2:
        print("\n" + "!" * 90 + "\n!! Only the regex labels were found. Attach pilkwang/rsna-knee-llm-labels (Add Input) -"
              "\n!! LLM labels are the single biggest lever on the score.\n" + "!" * 90 + "\n")
    Y[is_gold] = np.nan_to_num(gold[is_gold], nan=0.0); LM[is_gold] = (~np.isnan(gold[is_gold])).astype(np.float32)

    log("indexing cache files...")
    paths = find_cache_files()
    have = tr.StudyInstanceUID.isin(paths).values
    print(f"cached studies found: {have.sum()} / {len(tr)}")
    assert have.sum() > 0, "No cache found: run with MODE='full', or attach the knee_cache_s* datasets"

    weak_idx = np.where(~is_gold & have)[0]
    rare = [LABELS.index(l) for l in ("ACL", "MCL", "Fracture", "Medial OA")]
    strat = ((Y[weak_idx][:, rare] >= 0.5) * np.array([1, 2, 4, 8])).sum(1)
    fold_of = np.full(len(tr), -1)
    for k, (_, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=42).split(weak_idx, strat)):
        fold_of[weak_idx[va]] = k

    if USE_DISTILL:
        oofs = []
        for dirpath, _, filenames in walk_inputs():
            oofs += [pd.read_csv(Path(dirpath) / f) for f in filenames if re.match(r"oof_v\d+_f\d+\.csv$", f)]
        if oofs:
            oof = pd.concat(oofs).groupby("StudyInstanceUID")[LABELS].mean().reindex(tr.StudyInstanceUID)
            ok = oof.notna().all(1).values & ~is_gold
            Y[ok] = (1 - DISTILL_W) * Y[ok] + DISTILL_W * oof.values[ok]
            print(f"distillation: mixed OOF predictions into {ok.sum()} weak targets (w={DISTILL_W})")
        else:
            print("USE_DISTILL=True but no oof_v*_f*.csv (fold OOF) attached; training on fused labels only")
    return dict(uids=tr.StudyInstanceUID.values, Y=Y, LM=LM, gold=gold, is_gold=is_gold, paths=paths,
                have=have, fold_of=fold_of, refine_alpha=_refine_alpha(table))

def train_fold(fold, P, final_all=False, deadline=None, dev=None, n_workers=None):
    """Train one fold (or all data) on device `dev` and save its EMA weights. Stops before `deadline`."""
    dev = dev or TRAIN_DEVICES[0]
    is_cuda = dev.startswith("cuda")
    if is_cuda: torch.cuda.set_device(dev)                 # per-thread current device
    nw = NUM_WORKERS if n_workers is None else n_workers
    uids, Y, LM, gold, is_gold, paths, have, fold_of = (P[k] for k in
        ("uids", "Y", "LM", "gold", "is_gold", "paths", "have", "fold_of"))
    seed = 42 + SEED0 + (fold or 0)
    random.seed(seed); np.random.seed(seed)      # v12: torch generators are seeded per member under a lock, see below
    Wt = np.ones(len(uids), np.float32)
    all_weak = TRAIN_SCHEME == "all_weak" and not final_all
    if final_all:
        tri = np.where(have)[0]; Wt[is_gold] = GOLD_WEIGHT
        vai = np.where(is_gold & have)[0]                      # still reported, but it is training data now
    elif all_weak:                                             # 'fold' is the member index: all weak data, gold-only validation
        tri = np.where(have & ~is_gold)[0]
        vai = np.where(have & is_gold)[0]
    else:
        tri = np.where(have & ~is_gold & (fold_of != fold))[0]
        vai = np.where(have & (is_gold | (fold_of == fold)))[0]
    # per-epoch validation subset: all gold + a fixed random sample of weak studies
    rs = np.random.RandomState(seed)
    va_g, va_w = vai[is_gold[vai]], vai[~is_gold[vai]]
    vai_ep = np.sort(np.r_[va_g, rs.choice(va_w, min(VAL_WEAK_MAX, len(va_w)), replace=False)]) if len(va_w) else vai
    tag = "all" if final_all else (f"s{fold}" if all_weak else f"f{fold}")
    select_last = final_all or (all_weak and SELECT_LAST)
    L = lambda *a: log(f"[{tag}@{dev}]", *a)
    L(f"train {len(tri)} | val {len(vai)} (gold {int(is_gold[vai].sum())}; per-epoch {len(vai_ep)})")

    mk = lambda idx, train: CacheDS(uids[idx], paths, Y[idx].astype(np.float32), LM[idx].astype(np.float32), Wt[idx], train)
    to = 600 if nw > 0 else 0                              # a frozen worker raises after 10 min instead of hanging
    common = dict(num_workers=nw, pin_memory=is_cuda, timeout=to, persistent_workers=nw > 0,
                  prefetch_factor=4 if nw > 0 else None)
    dl_tr = DataLoader(mk(tri, True), batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **common)
    dl_ep = DataLoader(mk(vai_ep, False), batch_size=BATCH_SIZE * 2, shuffle=False, **common)

    L(f"building model {BACKBONE} (pretrained={PRETRAINED})...")
    with _SEED_LOCK:    # v12: torch.manual_seed reseeds EVERY GPU, so a second member starting up could reset the first one. Seed only our own CPU + GPU generator.
        torch.default_generator.manual_seed(seed)
        if is_cuda: torch.cuda.manual_seed(seed)
        model = KneeReader(pretrained=PRETRAINED).to(dev).to(memory_format=torch.channels_last)
    core = model
    L(f"model ready | batch {BATCH_SIZE} x accum {ACCUM} | {len(dl_tr)} iterations/epoch")
    enc_ids = {id(p) for p in core.enc.parameters()}
    groups = []
    for is_enc, lr in ((True, LR_ENC), (False, LR_HEAD)):
        ps = [p for p in core.parameters() if (id(p) in enc_ids) == is_enc]
        groups += [{"params": [p for p in ps if p.ndim > 1], "lr": lr, "weight_decay": WD},
                   {"params": [p for p in ps if p.ndim <= 1], "lr": lr, "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups)
    steps_per_epoch = max(1, len(dl_tr) // ACCUM)
    total, warm = EPOCHS * steps_per_epoch, max(1, int(WARMUP_EPOCHS * steps_per_epoch))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    scaler = torch.amp.GradScaler("cuda", enabled=is_cuda)
    ema = EMA(core, EMA_DECAY)
    eval_model = KneeReader(pretrained=False).to(dev).to(memory_format=torch.channels_last)
    ckpt_on = GRAD_CKPT

    prev = np.clip(np.nan_to_num(Y[tri]).mean(0), 1e-3, 0.999)
    pos_w = torch.tensor(np.clip((1 - prev) / prev, 1.0, max(POS_W_MAX, 1.0)), dtype=torch.float32, device=dev)
    L(f"windows per study: train {NV * (T_TRAIN or T_SLICES)} (random), eval {NV * T_SLICES} x {N_TTA} passes | "
      f"pos_weight {[round(float(v), 1) for v in pos_w.cpu()]}")
    refine_on = REFINE
    if refine_on:
        y_tri = torch.tensor(np.nan_to_num(Y[tri]), dtype=torch.float32, device=dev)
        mem = y_tri.clone()                       # per-study memory of the model's own past predictions; starts at the label
        weak_t = torch.tensor((~is_gold[tri]).astype(np.float32), device=dev)
        alpha_j = torch.tensor(np.asarray(P.get("refine_alpha", np.full(len(LABELS), REFINE_ALPHA)), np.float32), device=dev)
        L("label refinement ON, max memory share per finding: " + str({l: round(float(a), 2) for l, a in zip(LABELS, alpha_j.cpu().numpy())}))

    def step():
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(core.parameters(), 2.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); ema.update(core)

    def fwd_bwd(x, z, vm, lat, y, lm, w, ix, ramp, upd):
        logit = model(x, z, vm, lat)
        if refine_on:
            with torch.no_grad():
                t_old = mem[ix]
                a = (ramp * alpha_j)[None, :] * weak_t[ix][:, None]
                y = (1 - a) * y + a * t_old                       # target = label blended with the model's own running consensus
                if upd: mem[ix] = REFINE_BETA * t_old + (1 - REFINE_BETA) * torch.sigmoid(logit.detach().float())
        l = F.binary_cross_entropy_with_logits(logit.float(), y, reduction="none", pos_weight=pos_w) * lm * w[:, None]
        loss = l.sum() / lm.sum().clamp(min=1)
        scaler.scale(loss / ACCUM).backward()
        return loss.item()

    best, best_sd, hist, t_start = -1.0, None, [], time.time()
    n_ep = EPOCHS                                             # may shrink after timing the first iterations of epoch 0
    it_a, it_b = min(20, len(dl_tr) // 4), min(120, len(dl_tr) // 2)
    for ep in range(EPOCHS):
        if ep >= n_ep: break
        t_ep = time.time()
        model.train(); tl, n = 0.0, 0; opt.zero_grad(set_to_none=True)
        ramp = 0.0 if ep < REFINE_START else min(1.0, (ep - REFINE_START + 1) / REFINE_RAMP)
        upd = ep >= REFINE_FROM
        for it, (x, z, vm, lat, y, lm, w, ix) in enumerate(dl_tr):
            x = gpu_augment(x[:, 0].to(dev, non_blocking=True)).contiguous()
            z, vm, lat, y, lm, w, ix = [t.to(dev, non_blocking=True) for t in (z[:, 0], vm, lat, y, lm, w, ix)]
            z = z.contiguous()
            try:
                lv = fwd_bwd(x, z, vm, lat, y, lm, w, ix, ramp, upd)
            except torch.cuda.OutOfMemoryError:
                if ckpt_on: raise
                opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                core.enc.set_grad_checkpointing(True); ckpt_on = True
                L("out of GPU memory: gradient checkpointing switched ON, retrying")
                lv = fwd_bwd(x, z, vm, lat, y, lm, w, ix, ramp, upd)
            if (it + 1) % ACCUM == 0: step()
            tl += lv; n += 1
            if ep == 0 and it == it_a: t_a = time.time()
            if ep == 0 and it == it_b and deadline is not None and it_b > it_a:
                est_ep = (time.time() - t_a) / (it_b - it_a) * len(dl_tr) * 1.10    # +10% for the per-epoch evaluation
                fit = int((deadline - t_ep) // est_ep)
                if fit < n_ep:
                    n_ep = max(2, fit); total = n_ep * steps_per_epoch      # the LR lambda reads `total` at call time
                    L(f"   time budget: ~{est_ep/60:.0f} min/epoch -> training {n_ep} epochs (cosine schedule shortened to match)")
                    L(f"!! WARNING: the time budget cuts training from {EPOCHS} to {n_ep} epochs. Raise KNEE_BUDGET_H (max ~11), and build the cache "
                      f"in CPU sessions (MODE='cache') so the GPU session does not spend hours caching. An under-trained model is a likely score limiter.")
            if it < 3 or it % 100 == 0:
                spd = (it + 1) / max(time.time() - t_ep, 1e-6)
                L(f"ep{ep} it{it}/{len(dl_tr)} loss {tl/n:.4f} | {spd:.2f} it/s -> epoch ~{len(dl_tr)/spd/60:.0f} min")
        if n % ACCUM != 0: step()                                 # flush leftover accumulated gradients

        eval_model.load_state_dict(ema.state_dict())
        pv = predict(eval_model, dl_ep)
        g = is_gold[vai_ep]
        auc_w, _ = macro_auc(Y[vai_ep][~g], pv[~g], LM[vai_ep][~g]) if (~g).any() else (float("nan"), {})
        auc_g, per_g = macro_auc(gold[vai_ep][g], pv[g]) if g.any() else (float("nan"), {})
        score = float(np.nanmean([auc_g, auc_w]))
        ep_time = time.time() - t_ep
        hist.append(dict(epoch=ep, loss=tl / max(n, 1), auc_weak=auc_w, auc_gold=auc_g, score=score, minutes=ep_time / 60))
        L(f"epoch {ep}: loss {tl/max(n,1):.4f} | weak-val AUC {auc_w:.4f} | gold AUC {auc_g:.4f} | "
          f"score {score:.4f} | {ep_time/60:.1f} min")
        if refine_on: L(f"   refine: mix ramp {ramp:.2f} | mean |memory - label| {float((mem - y_tri).abs().mean()):.4f}")
        if per_g: L("   gold per finding: " + str({k: round(v, 3) for k, v in per_g.items()}))
        out_of_time = deadline is not None and time.time() + ep_time > deadline   # next epoch would not finish
        last = ep == n_ep - 1 or out_of_time
        if (not select_last and score > best) or select_last:   # select_last: overwrite every epoch, so a cut-off run still leaves weights
            best = score
            best_sd = {k: v.detach().cpu().clone() for k, v in ema.state_dict().items()}
            torch.save(best_sd, WORK / f"{WPFX}_{tag}.pt")      # v12: fp32 (v11 stored fp16 while its OOF/gold numbers came from fp32 weights)
            L(f"   saved {WPFX}_{tag}.pt  score {best:.4f}")
        if out_of_time and ep < n_ep - 1:
            L("   time budget: stopping this fold early"); break

    # out-of-fold predictions on the FULL validation set, once, with the best weights
    if best_sd is not None:
        eval_model.load_state_dict(best_sd)
        dl_va = DataLoader(mk(vai, False), batch_size=BATCH_SIZE * 2, shuffle=False,
                           **dict(common, persistent_workers=False))
        pv = predict(eval_model, dl_va)
        oof = pd.DataFrame(pv, columns=LABELS); oof.insert(0, "StudyInstanceUID", uids[vai])
        oof["is_gold"] = is_gold[vai]; oof.to_csv(WORK / f"oof_{VER}_{tag}.csv", index=False)
        g = is_gold[vai]
        auc_full, _ = macro_auc(Y[vai][~g], pv[~g], LM[vai][~g]) if (~g).any() else (float("nan"), {})
        auc_gold_full, _ = macro_auc(gold[vai][g], pv[g]) if g.any() else (float("nan"), {})
        L(f"final predictions on {len(vai)} validation studies: weak-val AUC {auc_full:.4f} | gold AUC {auc_gold_full:.4f}")
        del dl_va

    cfg = dict(VIEWS=VIEWS, SPAN=list(SPAN), SPACING_MM=SPACING_MM, IMG=IMG, DEPTH=DEPTH, T_SLICES=T_SLICES, BACKBONE=BACKBONE,
               DIM=DIM, MIX_LAYERS=MIX_LAYERS, HEADS=HEADS, LABELS=LABELS, CANONICALIZE_LATERALITY=CANONICALIZE_LATERALITY)
    json.dump(cfg, open(WORK / f"{WPFX}_config.json", "w"), indent=2)
    pd.DataFrame(hist).to_csv(WORK / f"history_{VER}_{tag}.csv", index=False)
    del model, eval_model, opt, ema, dl_tr, dl_ep
    gc.collect()
    if is_cuda: torch.cuda.empty_cache()
    return dict(fold=tag, device=dev, best=best, epochs=len(hist), planned=EPOCHS, minutes=(time.time() - t_start) / 60)

def run_folds(folds, P, deadline):
    """Train folds in rounds of len(TRAIN_DEVICES) (one per GPU, at the same time). A round only starts if the previous
    round's duration still fits before the deadline. If parallel training fails, the folds are retried one at a time."""
    import threading, traceback
    results, queue = [], list(folds)
    nw = max(2, NUM_WORKERS // len(TRAIN_DEVICES))
    while queue:
        left = deadline - time.time()
        need = max((r["minutes"] for r in results), default=0) * 60
        if results and need > left:
            log(f"skipping folds {queue}: a round needs ~{need/3600:.1f} h, {left/3600:.1f} h left"); break
        batch, queue = queue[:len(TRAIN_DEVICES)], queue[len(TRAIN_DEVICES):]
        if len(batch) == 1:
            results.append(train_fold(batch[0], P, deadline=deadline, dev=TRAIN_DEVICES[0])); continue
        log(f"training folds {batch} in parallel on {TRAIN_DEVICES[:len(batch)]}")
        out, errs = {}, {}
        def runner(f, d):
            try: out[f] = train_fold(f, P, deadline=deadline, dev=d, n_workers=nw)
            except Exception as e: errs[f] = traceback.format_exc()
        ths = [threading.Thread(target=runner, args=(f, d), daemon=True) for f, d in zip(batch, TRAIN_DEVICES)]
        for t in ths: t.start()
        for t in ths: t.join()
        results += [out[f] for f in batch if f in out]
        for f, tb in errs.items():
            log(f"fold {f} failed in parallel mode:\n{tb}\nretrying fold {f} alone on {TRAIN_DEVICES[0]}")
            results.append(train_fold(f, P, deadline=deadline, dev=TRAIN_DEVICES[0]))
    return results

def _gold_preds(ver):
    """Gold-study predictions of every oof_{ver}_*.csv found (this run's output first, then attached packs)."""
    files = {}
    for f in WORK.glob(f"oof_{ver}_*.csv"): files.setdefault(f.name, f)
    for dirpath, _, fns in walk_inputs():
        for f in fns:
            if re.match(rf"oof_{ver}_.*\.csv$", f): files.setdefault(f, Path(dirpath) / f)
    if not files: return None
    d = pd.concat([pd.read_csv(p) for p in files.values()])
    d["StudyInstanceUID"] = d.StudyInstanceUID.astype(str)
    return d[d.is_gold.astype(bool)].groupby("StudyInstanceUID")[LABELS].mean()

def paired_report(P, n_boot=1000):
    """v7 vs v9 vs v7+v9 on the gold studies, with a paired bootstrap for v9 - v7 (58 studies: wide interval, read it as such)."""
    a, b = _gold_preds("v7"), _gold_preds(VER)
    if a is None or b is None:
        print("paired report: attach the v7 pack (oof_v7_*.csv) to compare v9 with v7"); return
    gi = np.where(P["is_gold"])[0]; gold = pd.DataFrame(P["gold"][gi], index=P["uids"][gi], columns=LABELS)
    idx = [u for u in gold.index if u in a.index and u in b.index]
    G, A, B = gold.loc[idx].values, a.loc[idx].values, b.loc[idx].values
    macro = lambda p, ii: np.nanmean([gold_auc(G[ii, j], p[ii, j]) for j in range(len(LABELS))])
    allr = np.arange(len(idx)); ens = (A + B) / 2
    print(f"=== paired gold comparison on {len(idx)} studies ===")
    print(f"v7 alone {macro(A, allr):.4f} | {VER} alone {macro(B, allr):.4f} | v7+{VER} average {macro(ens, allr):.4f}")
    rng = np.random.default_rng(0); d_b, d_e = [], []
    for _ in range(n_boot):
        ii = rng.integers(0, len(idx), len(idx))
        d_b.append(macro(B, ii) - macro(A, ii)); d_e.append(macro(ens, ii) - macro(A, ii))
    for name, d in ((f"{VER} - v7", d_b), (f"v7+{VER} - v7", d_e)):
        lo, hi = np.nanpercentile(d, [2.5, 97.5]); print(f"   {name}: {np.nanmean(d):+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")

def _ens_versions():
    """This version plus the versions named in ENSEMBLE_WITH (e.g. knee_v11 -> v11)."""
    return list(dict.fromkeys([VER] + [m.group(1) for m in (re.match(r"knee_(v\d+)$", p) for p in ENSEMBLE_WITH) if m]))

def _all_oof_files(versions):
    """Every oof_{ver}_*.csv of these versions: this run's output first, then attached packs (one copy per file name)."""
    files = {}
    for ver in versions:
        for f in sorted(WORK.glob(f"oof_{ver}_*.csv")): files.setdefault(f.name, f)
        for dirpath, _, fns in walk_inputs():
            for f in fns:
                if re.match(rf"oof_{ver}_.*\.csv$", f): files.setdefault(f, Path(dirpath) / f)
    return list(files.values())

if MODE in ("train", "full"):
    P = prepare_training()
    deadline = T0 + (TIME_BUDGET_H - (FINAL_RESERVE_H if MODE == "full" else 0)) * 3600
    if MODE == "train":
        results = [train_fold(FOLD, P, final_all=FINAL_ALL_DATA, deadline=deadline)]
    else:
        results = run_folds(list(range(N_MEMBERS)) if TRAIN_SCHEME == "all_weak" else FULL_FOLDS, P, deadline)
    summary = pd.DataFrame(results); summary.to_csv(WORK / "training_summary.csv", index=False)
    print("\n=== training summary ==="); print(summary.round(4).to_string(index=False))
    if (summary.epochs < summary.planned).any():
        print("!! WARNING: " + str(summary[summary.epochs < summary.planned].fold.tolist()) + f" trained fewer than {EPOCHS} epochs (time budget). Expect a weaker model than a full run.")
    print(f"session time so far: {(time.time()-T0)/3600:.2f} h")
    # ensemble gold AUC (the number to compare against the stack): average every saved model's gold predictions
    _oof_files = _all_oof_files(_ens_versions())      # v12: includes the attached v11 members
    print("gold predictions of:", sorted(f.name for f in _oof_files))
    gp = [pd.read_csv(f) for f in _oof_files]
    if gp:
        gp = pd.concat(gp); gp = gp[gp.is_gold.astype(bool)].groupby("StudyInstanceUID")[LABELS].mean()
        gi = np.where(P["is_gold"])[0]; gidx = pd.Series(gi, index=P["uids"][gi])
        common = [u for u in gp.index if u in gidx.index]
        ens, per = macro_auc(P["gold"][gidx[common].values], gp.loc[common].values)
        print(f"ENSEMBLE gold AUC ({len(common)} gold studies): {ens:.4f}")
        print("   per finding:", {k: round(v, 3) for k, v in per.items()})
    try: paired_report(P)
    except Exception as e: print("paired report skipped:", repr(e))

## 7b. Save results as one zip: `kneemind_pack.zip`
Everything the offline submission run needs, in one file (the big cache is NOT included):
`knee_v7_*.pt` (models), `knee_v7_config.json`, `wheels/*.whl` (offline installs), training summaries.

* It lands in `/kaggle/working`, so it is part of this version's **Output** (download it from there if you want a copy).
* In the submission run, attach this notebook's output **or** upload the zip as a Dataset; either way the
  inference cells find and unzip it automatically.
* Safe to re-run on its own at any time after training.

In [ ]:
def make_pack():
    import zipfile
    out = WORK / PACK_NAME
    files = sorted(WORK.glob(f"{WPFX}_*.pt")) + [WORK / f"{WPFX}_config.json"]
    files += sorted((WORK / "wheels").glob("*.whl")) if (WORK / "wheels").exists() else []
    files += [WORK / f for f in ("training_summary.csv", "label_sources_vs_gold.csv") if (WORK / f).exists()]
    files += sorted(WORK.glob(f"history_{VER}_*.csv")) + sorted(WORK.glob(f"oof_{VER}_*.csv"))
    files = [f for f in files if f.exists()]
    if not any(f.name.startswith(WPFX + "_") and f.suffix == ".pt" for f in files):
        print("no trained weights in /kaggle/working yet: nothing to pack"); return None
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files: z.write(f, f.relative_to(WORK).as_posix())
    names = zipfile.ZipFile(out).namelist()
    print(f"saved {out}  ({out.stat().st_size/1e6:.0f} MB, {len(names)} files)")
    for n in names: print("   ", n)
    if not any(n.startswith("wheels/") for n in names):
        print("   WARNING: no wheels inside (was Internet OFF?). The offline run may not decode compressed DICOMs.")
    return out

if MODE in ("train", "full"):
    make_pack()

## 8. Inference engine (used by the hidden submission cell)
* Loads every `knee_v7_*.pt` from the attached training outputs (fold models and/or the all-data model) and averages them.
* Preprocessing is the exact `process_study` used to build the training cache, so train/test cannot drift.
* DICOM decoding runs in DataLoader worker processes while the GPU(s) run the previous batch, so neither waits on the other.
* 2-way slice TTA; it switches itself off if the projected runtime would pass 85% of `INFER_BUDGET_H`.
* A study that fails to decode gets 0.5 for every finding instead of crashing the run.
* A complete `submission.csv` (remaining studies at 0.5) is written every `CKPT_EVERY` studies, so even a timeout leaves a scoreable file.

In [ ]:
CKPT_EVERY = 100

def find_weights():
    if MODE == "full":                                    # a full run uses exactly the weights it just trained
        return sorted(WORK.glob(f"{WPFX}_*.pt"), key=lambda p: p.name)
    prefixes = [WPFX] + [p for p in ENSEMBLE_WITH if p != WPFX]
    found = {}                                            # one copy per model name (output files + pack copies)
    for dirpath, _, filenames in walk_inputs():
        for f in filenames:
            if any(re.match(pf + r"_(f\d+|s\d+|all)\.pt$", f) for pf in prefixes): found.setdefault(f, Path(dirpath) / f)
    for pf in prefixes:
        for p in WORK.glob(f"{pf}_*.pt"): found.setdefault(p.name, p)
    return [found[k] for k in sorted(found)]

class TestDS(Dataset):
    def __init__(self, uids, groups, root, n_tta):
        self.uids, self.groups, self.root, self.n_tta = uids, groups, root, n_tta
    def __len__(self): return len(self.uids)
    def __getitem__(self, i):
        uid = self.uids[i]; ok = 1.0
        try:
            vol, vm, lat = process_study(uid, self.groups[uid], self.root)
            if vm.sum() == 0: ok = 0.0
        except Exception:
            vol, vm, lat = np.zeros((NV, DEPTH, IMG, IMG), np.uint8), np.zeros(NV, np.float32), np.array([0, 0, 1], np.float32)
            ok = 0.0
        xs, zs = zip(*[make_input(vol, slice_idx(DEPTH, T_SLICES, k)) for k in range(self.n_tta)])
        return (torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(zs)), torch.from_numpy(vm),
                torch.from_numpy(lat), torch.tensor(ok), i)

def submission_template():
    """Study order and column order exactly as Kaggle expects them."""
    te = pd.read_csv(COMP_DIR / "test.csv"); te["StudyInstanceUID"] = te.StudyInstanceUID.astype(str)
    ss_path = COMP_DIR / "sample_submission.csv"
    cols = list(pd.read_csv(ss_path, nrows=0).columns) if ss_path.exists() else ["StudyInstanceUID"] + LABELS
    tmpl = pd.DataFrame({"StudyInstanceUID": te.StudyInstanceUID.drop_duplicates().values})
    for c in cols[1:]: tmpl[c] = 0.5
    return tmpl[cols]

SAME_AS_PRIMARY = ["VIEWS", "SPACING_MM", "IMG", "DEPTH", "T_SLICES", "CANONICALIZE_LATERALITY", "DIM", "MIX_LAYERS", "HEADS", "LABELS"]

def _cfg_of(prefix, w_paths):
    own = [p for p in w_paths if p.name.startswith(prefix + "_")]
    cp = own[0].parent / f"{prefix}_config.json"
    if not cp.exists(): cp = find_in_inputs(f"{prefix}_config.json")
    return json.load(open(cp)) if cp is not None and Path(cp).exists() else None

def load_models():
    global VIEWS, NV, SPAN, SPACING_MM, IMG, DEPTH, T_SLICES, BACKBONE, DIM, MIX_LAYERS, HEADS, CANONICALIZE_LATERALITY, GRAD_CKPT
    w_paths = find_weights()
    assert w_paths, f"No {WPFX}_*.pt found: attach your training notebook outputs with Add Input"
    print("weights found:", [str(p) for p in w_paths])
    prefixes = list(dict.fromkeys(re.match(r"(knee_v\d+)_", p.name).group(1) for p in w_paths))
    primary = WPFX if WPFX in prefixes else prefixes[0]
    cfgs = {pf: _cfg_of(pf, w_paths) for pf in prefixes}
    cfg = cfgs[primary]
    assert cfg is not None, f"{primary}_config.json not found next to the weights"
    VIEWS = [tuple(v) for v in cfg["VIEWS"]]; NV = len(VIEWS)
    SPACING_MM, IMG, DEPTH, T_SLICES = cfg["SPACING_MM"], cfg["IMG"], cfg["DEPTH"], cfg["T_SLICES"]
    BACKBONE, DIM, MIX_LAYERS, HEADS = cfg["BACKBONE"], cfg["DIM"], cfg["MIX_LAYERS"], cfg["HEADS"]
    CANONICALIZE_LATERALITY = cfg["CANONICALIZE_LATERALITY"]; GRAD_CKPT = False
    SPAN = tuple(cfg.get("SPAN", (0.0, 1.0)))     # v7 configs have no SPAN: full stack
    print(f"config ({primary}): {IMG}px @ {SPACING_MM} mm, depth {DEPTH}, T {T_SLICES}, {NV} views, span {SPAN}, {BACKBONE}")
    models, used = [], []
    for pf in [primary] + [p for p in prefixes if p != primary]:
        c = cfgs[pf]
        if pf != primary:
            bad = [k for k in SAME_AS_PRIMARY if c is None or c.get(k) != cfg.get(k)]
            if c is not None and list(c.get("SPAN", (0.0, 1.0))) != list(SPAN): bad.append("SPAN")
            if c is None or bad:
                print(f"SKIPPING {pf}: its preprocessing/architecture differs from {primary} ({'config missing' if c is None else bad}); "
                      f"mixing it in would feed it the wrong pixels"); continue
        for wp in [p for p in w_paths if p.name.startswith(pf + "_")]:
            m = KneeReader(backbone=c["BACKBONE"], pretrained=False, n_views=NV, n_out=len(LABELS))
            sd = torch.load(wp, map_location="cpu")
            m.load_state_dict({k: v.float() if v.dtype == torch.float16 else v for k, v in sd.items()})
            d = TRAIN_DEVICES[len(models) % len(TRAIN_DEVICES)] if DEVICE == "cuda" else "cpu"
            models.append(m.to(d).to(memory_format=torch.channels_last).eval()); used.append(wp)
    print(f"ensemble of {len(models)}:", [p.name for p in used])
    return models

def decode_selftest(groups, uids, root, n=12):
    """v10: decode the first test studies in this process and report how many chosen series came out usable."""
    ok = tot = 0
    for u in uids[:n]:
        try:
            _, vm, _ = process_study(u, groups[u], root)
            ok += int(vm.sum()); tot += len(pick_series(groups[u]))
        except Exception as e:
            print("decode self-test: study failed:", repr(e)[:160])
    print(f"decode self-test: {ok}/{tot} chosen series decoded on the first {min(n, len(uids))} test studies")
    if tot and ok / tot < 0.8:
        msg = (f"only {ok}/{tot} series decode: attach the training output that contains 'wheels/' (JPEG decoders) and check "
               f"the readiness block above; predictions would come from empty views")
        if not IS_RERUN: raise RuntimeError(msg)
        print("WARNING:", msg)

def run_inference(tmpl, ckpt_path, deadline=None):
    """deadline (epoch seconds): if passed before all studies are done, return None (caller falls back to the stack)."""
    models = load_models()
    tes = pd.read_csv(COMP_DIR / "test_series.csv")
    tes["StudyInstanceUID"] = tes.StudyInstanceUID.astype(str); tes["SeriesInstanceUID"] = tes.SeriesInstanceUID.astype(str)
    groups = {k: g for k, g in tes.groupby("StudyInstanceUID")}
    uids = tmpl.StudyInstanceUID.tolist()
    groups.update({u: tes.iloc[:0] for u in uids if u not in groups})
    decode_selftest(groups, uids, COMP_DIR / "test_series")
    P = np.full((len(uids), len(LABELS)), 0.5, np.float32); bad = 0
    dl = DataLoader(TestDS(uids, groups, COMP_DIR / "test_series", N_TTA), batch_size=2 * max(1, N_GPU),
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda", prefetch_factor=4)
    t, done, n_tta, next_ckpt = time.time(), 0, N_TTA, CKPT_EVERY
    budget_s = (deadline - t) if deadline is not None else INFER_BUDGET_H * 3600
    for b in dl:
        idx, ok = b[5].numpy(), b[4].numpy()
        p = predict(models, [b[:4]], max_k=n_tta)
        p[ok == 0] = 0.5; bad += int((ok == 0).sum())
        P[idx] = p; done += len(idx)
        el = time.time() - t
        if deadline is not None and time.time() > deadline:
            print(f"hard time limit reached after {done}/{len(uids)} studies: dropping this model's predictions"); return None
        if n_tta > 1 and done >= 40 and el / done * len(uids) > budget_s * 0.85:
            print(f"projected {el/done*len(uids)/3600:.1f} h > budget: slice TTA off from here"); n_tta = 1
        if done >= next_ckpt:
            out = tmpl.copy(); out[LABELS] = P; out.to_csv(ckpt_path, index=False); next_ckpt += CKPT_EVERY
            print(f"  {done}/{len(uids)} studies  {el/60:.1f} min  (ETA {(el/done)*(len(uids)-done)/60:.0f} min)", flush=True)
    print(f"test inference: {done} studies, {bad} undecodable (0.5), {(time.time()-t)/60:.1f} min")
    if bad > 0.5 * max(done, 1):
        msg = f"{bad}/{done} test studies could not be processed: check decoders (wheels) and the attached weights/config"
        if not IS_RERUN: raise RuntimeError(msg)
        print("WARNING:", msg)
    out = tmpl.copy(); out[LABELS] = P
    return out

## 9. Hidden submission
This is the cell that produces the scored file. When you press **Submit**, Kaggle re-runs the notebook on the full
hidden test set; `IS_RERUN` forces `MODE="infer"` so nothing else (caching, training) runs.

1. Predict every test study (section 8), writing checkpoints as it goes.
2. If `stack_submission.csv` exists in `/kaggle/working` (written by public-stack cells placed above this one),
   rank-blend: AUC only depends on ranking, so each source becomes per-finding percentile ranks first.
   The weight is chosen on the 58 gold studies when `stack_gold_preds.csv` (the stack's predictions on them, made
   with `gold_mirror_cell.py`) is attached; otherwise it is 0 (stack only) unless you set `W_MINE` yourself.
3. Validate against `sample_submission.csv`: same studies, same columns in the same order, no NaN, values in [0, 1].
4. Write `/kaggle/working/submission.csv`.

**Blending with a public stack**
1. Fork the public stack notebook. In a copy of it, paste `gold_mirror_cell.py` as the first cell and point the
   notebook's competition-data path at the mirror it builds; run it and save its `submission.csv` as a Dataset
   file named `stack_gold_preds.csv`.
2. In the fork itself, add `import shutil; shutil.copy('/kaggle/working/submission.csv', '/kaggle/working/stack_submission.csv')`
   after its last cell, then paste all of this notebook's code cells below it with `MODE="infer"`.
3. Attach this notebook's run-1 output and `stack_gold_preds.csv`; the last cell prints the gold table and the chosen weight.

**Final-submission safeguards (v7)**
* `stack_submission.csv` is written as `submission.csv` before anything else, so the stack alone is always the fallback.
* Weight 0 from the gate → this model's inference is **skipped** (no extra runtime).
* This model must finish by `HARD_LIMIT_H` (8.5 h) after the kernel started, counting the stack's own runtime;
  otherwise its predictions are dropped and the stack is submitted alone. A crash in this model does the same.

In [ ]:
# ============================== PROFILE (MODE="profile"): where does the time go? ==============================
# ~8 min on the real Kaggle GPUs, no training. Set MODE = "profile" in cell 1, GPU T4 x2, attach the competition data
# (and, optionally, the knee_cache_* outputs so the loader test reads real, compressed studies). Internet may be on or off.
# Prints three verdict lines: TRAINING bottleneck, INFERENCE bottleneck, and the projected hidden-run inference time.
if MODE == "profile":
    import threading, statistics, tempfile
    N_IT = int(os.environ.get("KNEE_PROFILE_ITERS", 12))
    N_STUDY = int(os.environ.get("KNEE_PROFILE_STUDIES", 32))
    EPOCH_STUDIES, TEST_STUDIES = 4349, 1300
    WIN = NV * T_SLICES
    print(f"PROFILE preset={PRESET}: {IMG}px, {NV} views x {T_SLICES} slices = {WIN} windows/study, {BACKBONE}, "
          f"batch {BATCH_SIZE}, {N_GPU} GPU(s), {os.cpu_count()} CPUs")

    def _sync(dev):
        if str(dev).startswith("cuda"): torch.cuda.synchronize(dev)

    def gpu_step_bench(dev, ckpt, n_it=N_IT):
        """(median s per training iteration of BATCH_SIZE studies, peak GB) on `dev`, or None on out-of-memory.
        One iteration = GPU augment + forward + backward + AdamW step (slightly pessimistic: real runs step every ACCUM iterations)."""
        global GRAD_CKPT
        is_cuda = str(dev).startswith("cuda")
        old, GRAD_CKPT = GRAD_CKPT, ckpt
        try: m = KneeReader(pretrained=False).to(dev).to(memory_format=torch.channels_last)
        finally: GRAD_CKPT = old
        m.train(); opt = torch.optim.AdamW(m.parameters(), lr=1e-5); scaler = torch.amp.GradScaler("cuda", enabled=is_cuda)
        B = BATCH_SIZE; TT = T_TRAIN or T_SLICES
        x = torch.randint(0, 255, (B, NV, TT, 3, IMG, IMG), dtype=torch.uint8, device=dev)
        z = torch.rand(B, NV, TT, device=dev); vm = torch.ones(B, NV, device=dev)
        lat = torch.zeros(B, 3, device=dev); lat[:, 1] = 1; y = torch.rand(B, len(LABELS), device=dev)
        if is_cuda: torch.cuda.reset_peak_memory_stats(dev)
        ts, out = [], None
        try:
            for i in range(n_it + 3):
                t = time.time()
                loss = F.binary_cross_entropy_with_logits(m(gpu_augment(x).contiguous(), z, vm, lat).float(), y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); _sync(dev)
                if i >= 3: ts.append(time.time() - t)
            out = (statistics.median(ts), torch.cuda.max_memory_allocated(dev) / 1e9 if is_cuda else float("nan"))
        except torch.cuda.OutOfMemoryError:
            out = None
        del m, opt, x
        gc.collect()
        if is_cuda: torch.cuda.empty_cache()
        return out

    def loader_bench(nw, n=N_STUDY):
        """Studies per second one training DataLoader with `nw` workers can deliver (real cache if attached, else synthetic)."""
        paths = find_cache_files(); uids = list(paths)[:n]; src = "real cache"
        if not uids:
            src = "SYNTHETIC cache (attach knee_cache_* for the real, compressed one)"
            d = Path(tempfile.mkdtemp()); yy, xx = np.mgrid[0:IMG, 0:IMG]
            base = (200 * np.exp(-(((xx - IMG / 2) / (IMG * .28)) ** 2 + ((yy - IMG / 2) / (IMG * .33)) ** 2))).astype(np.uint8)
            vol = np.stack([[np.clip(base + np.random.randint(0, 40, (IMG, IMG)), 0, 255) for _ in range(DEPTH)] for _ in range(NV)]).astype(np.uint8)
            paths = {}
            for i in range(n):
                paths[f"s{i}"] = d / f"s{i}.npz"; np.savez_compressed(paths[f"s{i}"], vol=vol, vmask=np.ones(NV, np.float32), lat=np.array([0, 1, 0], np.float32))
            uids = list(paths)
        k = len(uids); ds = CacheDS(uids, paths, np.zeros((k, 12), np.float32), np.ones((k, 12), np.float32), np.ones(k, np.float32), True)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, num_workers=nw, shuffle=False, drop_last=True)
        t0 = t1 = None; got = 0
        for i, b in enumerate(dl):
            if i == 0: t1 = time.time()
            else: got += len(b[0])
        return got / max(time.time() - t1, 1e-6), src

    _PS = {}
    def _decode_one(uid):
        t = time.time(); process_study(uid, _PS[uid], COMP_DIR / "train_series"); return time.time() - t

    def decode_bench(nproc, offset, n=N_STUDY):
        """Studies per second `nproc` worker processes can decode+preprocess from the competition DICOMs (cold files)."""
        global _PS
        trs = pd.read_csv(COMP_DIR / "train_series.csv")
        trs["StudyInstanceUID"] = trs.StudyInstanceUID.astype(str); trs["SeriesInstanceUID"] = trs.SeriesInstanceUID.astype(str)
        groups = {k: g for k, g in trs.groupby("StudyInstanceUID")}; uids = sorted(groups)[offset:offset + n]; _PS = {u: groups[u] for u in uids}
        t = time.time()
        with Pool(nproc) as pool: per = list(pool.imap_unordered(_decode_one, uids))
        return len(uids) / (time.time() - t), statistics.mean(per)

    def fwd_bench(dev, n_it=N_IT):
        """Seconds per study for inference forward (fp16, no grad) at the inference batch size."""
        is_cuda = str(dev).startswith("cuda"); B = 2 * max(1, N_GPU)
        m = KneeReader(pretrained=False).to(dev).to(memory_format=torch.channels_last).eval()
        x = torch.randint(0, 255, (B, NV, T_SLICES, 3, IMG, IMG), dtype=torch.uint8, device=dev)
        z = torch.rand(B, NV, T_SLICES, device=dev); vm = torch.ones(B, NV, device=dev); lat = torch.zeros(B, 3, device=dev)
        ts = []
        with torch.no_grad():
            for i in range(n_it + 2):
                t = time.time(); m(x, z, vm, lat); _sync(dev)
                if i >= 2: ts.append(time.time() - t)
        del m, x; gc.collect()
        return statistics.median(ts) / B

    # ---------------- 1. GPU training step ----------------
    dev0 = TRAIN_DEVICES[0]; steps = {}
    for ck in (False, True):
        r = gpu_step_bench(dev0, ck)
        steps[ck] = r
        if r is None: print(f"[gpu step] grad-ckpt {'on ' if ck else 'off'}: OUT OF MEMORY at batch {BATCH_SIZE} x {WIN} windows")
        else: print(f"[gpu step] grad-ckpt {'on ' if ck else 'off'}: {r[0]:.3f} s/it = {BATCH_SIZE / r[0]:.1f} studies/s | peak {r[1]:.1f} GB "
                    f"| ~{r[0] * EPOCH_STUDIES / BATCH_SIZE / 60 * 1.1:.0f} min/epoch (one model, +10% eval)")
    ok = {k: v for k, v in steps.items() if v is not None}
    best_ck = min(ok, key=lambda k: ok[k][0]) if ok else None
    gpu_rate = BATCH_SIZE / ok[best_ck][0] if ok else float("nan")
    if best_ck is not None and len(TRAIN_DEVICES) > 1:
        res = {}
        def _w(d): res[d] = gpu_step_bench(d, best_ck)
        ths = [threading.Thread(target=_w, args=(d,)) for d in TRAIN_DEVICES]
        [t.start() for t in ths]; [t.join() for t in ths]
        par = [v[0] for v in res.values() if v is not None]
        if par:
            slow = statistics.mean(par) / ok[best_ck][0]
            print(f"[gpu step] both GPUs at once (2 threads, one process, like the training run): {statistics.mean(par):.3f} s/it each "
                  f"= x{slow:.2f} vs one GPU alone -> {'thread/GIL/CPU contention' if slow > 1.25 else 'no meaningful contention'}")
            gpu_rate = BATCH_SIZE / statistics.mean(par)

    # ---------------- 2. training data loader ----------------
    nw_train = max(2, NUM_WORKERS // max(1, len(TRAIN_DEVICES)))
    ld_rate, ld_src = loader_bench(nw_train)
    print(f"[loader ] {nw_train} workers (what each model gets): {ld_rate:.1f} studies/s from {ld_src}")

    # ---------------- 3. inference: DICOM decode vs GPU forward ----------------
    n_mem = len(find_weights()) or N_MEMBERS          # members run one after another in predict(), so their costs add up
    fwd_s = fwd_bench(dev0); fwd_rate = 1.0 / (n_mem * max(1, N_TTA) * fwd_s)
    print(f"[fwd    ] {fwd_s * 1000:.0f} ms/study per model per TTA pass -> {fwd_rate:.1f} studies/s with {n_mem} members x {N_TTA} TTA pass(es)")
    dec = {}
    if (COMP_DIR / "train_series.csv").exists():
        for i, nproc in enumerate(sorted({NUM_WORKERS, 2 * (os.cpu_count() or 4)})):
            rate, per = decode_bench(nproc, offset=i * N_STUDY)
            dec[nproc] = rate; print(f"[decode ] {nproc:2d} processes: {rate:.2f} studies/s ({per:.2f} s per study per process, cold files)")
    else:
        print("[decode ] competition data not attached: skipped")

    # ---------------- verdicts ----------------
    print("\n=== VERDICT ===")
    print(f"TRAINING : per model GPU {gpu_rate:.1f} studies/s vs data loader {ld_rate:.1f} studies/s -> "
          f"bottleneck = {'GPU' if gpu_rate <= ld_rate else 'DATA LOADER (use uncompressed npz, more workers, or fewer parallel models)'}")
    if dec:
        best_dec = max(dec.values())
        print(f"INFERENCE: decode {best_dec:.2f} studies/s vs GPU forward {fwd_rate:.1f} studies/s -> bottleneck = "
              f"{'DICOM decode (CPU)' if best_dec <= fwd_rate else 'GPU'}; projected {TEST_STUDIES / min(best_dec, fwd_rate) / 60:.0f} min for {TEST_STUDIES} test studies "
              f"(+ ~2 min start-up). Each extra 12 min is worth 0.01 AUC in the efficiency score.")
        if len(dec) > 1:
            lo, hi = min(dec), max(dec); print(f"           {hi} processes vs {lo}: x{dec[hi] / dec[lo]:.2f} -> "
                                               f"{'raise the inference DataLoader num_workers' if dec[hi] / dec[lo] > 1.15 else 'worker count is not the limit'}")


In [ ]:
def _find_file(name):
    p = WORK / name
    return p if p.exists() else find_in_inputs(name)

def rank_blend(a, b, w):
    """a, b: [N, 12] arrays; per-finding percentile ranks, weight w on a."""
    ra = pd.DataFrame(a).rank(pct=True).values; rb = pd.DataFrame(b).rank(pct=True).values
    return w * ra + (1 - w) * rb

def choose_blend_weight():
    """One global weight for this model, chosen on the 58 gold studies (never trained on by this model).
    Returns 0 unless the blend beats the stack alone by GATE_MIN_GAIN."""
    sg_path = _find_file("stack_gold_preds.csv")
    oofs = _all_oof_files(_ens_versions())            # v12: this version's and the ensembled versions' gold predictions
    if sg_path is None or not oofs:
        print("blend gate: stack_gold_preds.csv or this model's gold predictions (oof_v*_*.csv) not found -> weight 0 "
              "(stack only). Set W_MINE to force a weight.")
        return 0.0
    tr = pd.read_csv(COMP_DIR / "train.csv"); tr["StudyInstanceUID"] = tr.StudyInstanceUID.astype(str)
    g = tr.set_index("StudyInstanceUID")[LABELS].apply(pd.to_numeric, errors="coerce")
    g = g[g.notna().any(axis=1)]
    mine_g = pd.concat([pd.read_csv(p) for p in dict.fromkeys(oofs)])
    mine_g["StudyInstanceUID"] = mine_g.StudyInstanceUID.astype(str)
    mine_g = mine_g[mine_g.is_gold.astype(bool)].groupby("StudyInstanceUID")[LABELS].mean()
    st = pd.read_csv(sg_path); st["StudyInstanceUID"] = st.StudyInstanceUID.astype(str)
    st = st.groupby("StudyInstanceUID")[LABELS].mean()
    common = [u for u in g.index if u in mine_g.index and u in st.index]
    if len(common) < min(40, len(g)):
        print(f"blend gate: only {len(common)} gold studies have both predictions -> weight 0"); return 0.0
    G, M, S = g.loc[common].values, mine_g.loc[common].values, st.loc[common].values
    macro = lambda p: np.nanmean([gold_auc(G[:, j], p[:, j]) for j in range(len(LABELS))])
    print(f"blend gate on {len(common)} gold studies: this model alone {macro(M):.4f} | stack alone {macro(S):.4f}")
    res = {w: macro(rank_blend(M, S, w)) for w in W_GRID}
    for w, a in res.items(): print(f"   w={w:.2f}  gold macro AUC {a:.4f}")
    best_w = max(res, key=lambda w: (round(res[w], 6), -w))     # ties -> smaller weight
    if res[best_w] - res[0.0] < GATE_MIN_GAIN:
        print(f"   best gain {res[best_w] - res[0.0]:+.4f} < {GATE_MIN_GAIN}: weight 0 (stack only)"); return 0.0
    print(f"   chosen weight {best_w} (gain {res[best_w] - res[0.0]:+.4f}; gold-58 SE is ~0.02, so treat small gains with care)")
    return best_w

HARD_LIMIT_H = float(os.environ.get("KNEE_HARD_LIMIT_H", 8.5))   # Kaggle kills the hidden run at 9 h
def session_start():
    """When this kernel started (stack cells above this notebook count against the same 9 h)."""
    try:
        import psutil; return psutil.Process(os.getpid()).create_time()
    except Exception:
        return T0

if MODE in ("infer", "full"):
    SUB_PATH = WORK / "submission.csv"
    print("=== offline readiness ===")
    print(" internet       :", "ON (fine for this check; turn it OFF in the submission run)" if ONLINE else "OFF (required for submission)")
    print(" weights        :", [p.name for p in find_weights()] or "NONE: attach training outputs")
    print(" decoders       :", DECODERS)
    print(" hidden re-run  :", IS_RERUN)
    tmpl = submission_template()
    stack_path = WORK / "stack_submission.csv"
    stack_sub, w = None, None
    if stack_path.exists():                                 # FINAL mode: public stack above, this model blended in
        st = pd.read_csv(stack_path); st["StudyInstanceUID"] = st.StudyInstanceUID.astype(str)
        st = st.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID").reindex(tmpl.StudyInstanceUID)
        stack_sub = tmpl.copy(); stack_sub[LABELS] = st[LABELS].astype(float).fillna(0.5).values
        stack_sub.to_csv(SUB_PATH, index=False)             # the stack alone is the fallback from the first second
        _w_env = os.environ.get("KNEE_W_MINE")
        w = float(_w_env) if _w_env else (choose_blend_weight() if W_MINE is None else float(W_MINE))
        print(f"stack found ({int(st[LABELS[0]].notna().sum())}/{len(tmpl)} studies); weight of this model: {w}")
    else:
        tmpl.to_csv(SUB_PATH, index=False)                  # a valid file exists from the first second

    mine = None
    if stack_sub is None or w > 0:
        deadline = session_start() + HARD_LIMIT_H * 3600 if stack_sub is not None else None
        if deadline is not None:
            print(f"time used so far this session: {(time.time() - session_start())/3600:.2f} h; "
                  f"this model must finish within {(deadline - time.time())/3600:.2f} h or the stack is submitted alone")
        ckpt = SUB_PATH if stack_sub is None else WORK / f"submission_{VER}_partial.csv"
        try:
            mine = run_inference(tmpl, ckpt, deadline)
        except Exception:
            import traceback; traceback.print_exc()
            if not IS_RERUN and stack_sub is None: raise    # interactive, own model only: fail loudly
            if stack_sub is None:
                mine = pd.read_csv(SUB_PATH); print("inference failed: submitting checkpointed predictions")
            else:
                print("this model failed: submitting the stack alone")
        if mine is not None: mine.to_csv(WORK / f"submission_{VER}.csv", index=False)
    else:
        print("weight 0: skipping this model's inference entirely (saves its runtime), submitting the stack alone")

    if stack_sub is None:
        sub = mine.copy()
    elif mine is None:
        sub = stack_sub.copy()
    else:
        sub = tmpl.copy()
        sub[LABELS] = rank_blend(mine[LABELS].values, stack_sub[LABELS].values, w)
        print(f"rank-blended: {w} x this model + {1 - w} x stack")

    # ---- validation ----
    sub = sub[tmpl.columns]
    sub[LABELS] = sub[LABELS].astype(float).fillna(0.5).clip(0, 1)
    assert len(sub) == len(tmpl) and (sub.StudyInstanceUID.values == tmpl.StudyInstanceUID.values).all(), "study ids/order mismatch"
    assert list(sub.columns) == list(tmpl.columns), "column mismatch"
    assert not sub[LABELS].isna().any().any()
    sub.to_csv(SUB_PATH, index=False)
    print(f"submission.csv written: {sub.shape[0]} studies x {len(LABELS)} findings | session time {(time.time() - session_start())/3600:.2f} h")
    print("per-finding mean prediction:", sub[LABELS].mean().round(3).to_dict())
    print(sub.head())